# IMPLEMENTASI DAN PERBANDINGAN ALGORITMA RANDOM FOREST DAN K-NEAREST NEIGHBOR DALAM KLASIFIKASI STATUS GIZI BALITA

Notebook ini digunakan untuk melakukan proses eksplorasi data, preprocessing, penanganan missing value, seleksi atau reduksi fitur, penanganan ketidakseimbangan kelas, pemodelan menggunakan algoritma Random Forest dan K-Nearest Neighbor, serta evaluasi performa model klasifikasi status gizi balita.

# 1. Data Acquisition


## 1.1 Menghubungkan Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 1.2 Import Library

In [ ]:
# ==========================================
# Library Manipulasi Data
# ==========================================
import pandas as pd
import numpy as np

# ==========================================
# Library Visualisasi
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Library Preprocessing
# ==========================================
from sklearn.preprocessing import LabelEncoder

# ==========================================
# Library Model Selection
# ==========================================
from sklearn.model_selection import train_test_split

# ==========================================
# Library Imbalanced Learning
# ==========================================
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# ==========================================
# Library Machine Learning
# ==========================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# ==========================================
# Library Evaluasi
# ==========================================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

# ==========================================
# Library Penyimpanan Model
# ==========================================
import joblib

print("Seluruh library berhasil diimpor.")

## 1.3 Memuat Dataset Penelitian

In [ ]:
# Lokasi dataset penelitian
file_path = "/content/drive/MyDrive/DATASET_TA/dataset_raw.xlsx"

# Membaca dataset
df = pd.read_excel(file_path)

print("Dataset berhasil dimuat.")

# 2. Data Exploration

## 2.1 Menampilkan Dataset

In [ ]:
# Menampilkan lima baris pertama dataset

df.head()

## 2.2 Menampilkan Informasi Dataset

In [ ]:
# Informasi dataset

df.info()

## 2.3 Menampilkan Dimensi Dataset

In [ ]:
# Menampilkan dimensi dataset

print(f"Jumlah Baris : {df.shape[0]}")
print(f"Jumlah Kolom : {df.shape[1]}")

## 2.4 Identifikasi Variabel Dataset

In [ ]:
# Menampilkan seluruh nama variabel

print("Daftar Variabel Dataset:\n")

for i, kolom in enumerate(df.columns, start=1):
    print(f"{i}. {kolom}")

## 2.5 Statistik Deskriptif

In [ ]:
# Statistik deskriptif variabel numerik

df.describe().T

## 2.6 Pemeriksaan Tipe Data Variabel

In [ ]:
# Menampilkan tipe data setiap variabel

pd.DataFrame({
    "Variabel": df.dtypes.index,
    "Tipe Data": df.dtypes.values
})

## 2.7 Analisis Distribusi Variabel Target (TB/U)

In [ ]:
# Menampilkan distribusi kategori pada variabel TB/U

distribusi_tbu = df["TB/U"].value_counts().reset_index()
distribusi_tbu.columns = ["Kategori TB/U", "Jumlah Data"]

display(distribusi_tbu)

## 2.8 Visualisasi Distribusi Variabel Target (TB/U)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="TB/U",
    order=df["TB/U"].value_counts().index,
    hue="TB/U",
    palette="viridis",
    legend=False
)

plt.title("Distribusi Kategori Status Gizi Berdasarkan TB/U")
plt.xlabel("Kategori TB/U")
plt.ylabel("Jumlah Data")

for p in plt.gca().patches:
    plt.text(
        p.get_x()+p.get_width()/2,
        p.get_height()+20,
        int(p.get_height()),
        ha="center",
        fontsize=10
    )

plt.show()

## 2.9 Identifikasi Variabel Numerik dan Kategorikal

In [ ]:
# Menghapus variabel identifier (No)
df = df.drop(columns=["No"])

# Mengelompokkan variabel berdasarkan tipe data
numerical_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = df.select_dtypes(include=["object"]).columns.tolist()

print("Variabel Numerik")
print("-"*50)
for col in numerical_features:
    print(col)

print("\nVariabel Kategorikal")
print("-"*50)
for col in categorical_features:
    print(col)

## 2.10 Visualisasi Distribusi Variabel Numerik

In [ ]:
# Visualisasi distribusi seluruh variabel numerik

plt.figure(figsize=(18,12))

df[numerical_features].hist(
    figsize=(18,12),
    bins=20,
    edgecolor="black",
    color="steelblue"
)

plt.suptitle(
    "Distribusi Variabel Numerik",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# ==========================
# Simpan gambar
# ==========================
plt.savefig(
    "Distribusi_Variabel_Numerik.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from google.colab import files

files.download("Distribusi_Variabel_Numerik.png")

## 2.11 Visualisasi Outlier Menggunakan Boxplot

In [ ]:
# ==========================================
# Visualisasi Outlier Menggunakan Boxplot
# ==========================================

fig, axes = plt.subplots(3, 3, figsize=(18, 12))

axes = axes.flatten()

for i, col in enumerate(numerical_features):

    sns.boxplot(
        y=df[col],
        ax=axes[i],
        color="skyblue"
    )

    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")

plt.suptitle(
    "Visualisasi Outlier Variabel Numerik",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout(rect=[0,0,1,0.96])

# ==========================================
# Simpan Gambar
# ==========================================

plt.savefig(
    "Boxplot_Variabel_Numerik.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from google.colab import files

files.download("Boxplot_Variabel_Numerik.png")

# 3. Data Preparation

## 3.1 Konversi Missing Value

In [ ]:
# ==========================================
# Konversi Missing Value Menjadi NaN
# ==========================================

import numpy as np

# Mengganti berbagai bentuk data kosong menjadi NaN
df = df.replace(
    ["", " ", "-", "--", "NULL", "null", "Null"],
    np.nan
)

print("Konversi missing value selesai dilakukan.")

## 3.2 Pemeriksaan Missing Value

In [ ]:
# ==========================================
# Pemeriksaan Missing Value
# ==========================================

missing_value = pd.DataFrame({
    "Jumlah Missing": df.isnull().sum(),
    "Persentase Missing (%)": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing_value = (
    missing_value[
        missing_value["Jumlah Missing"] > 0
    ]
    .sort_values(
        by="Persentase Missing (%)",
        ascending=False
    )
)

print(f"Jumlah seluruh data : {len(df)}")

display(missing_value)

## 3.3 Identifikasi Tipe Data Variabel dengan Missing Value

In [ ]:
# ==========================================
# Identifikasi Tipe Data Variabel Missing
# ==========================================

for kolom in missing_value.index:

    print(f"{kolom}")
    print(f"Tipe Data : {df[kolom].dtype}")
    print(df[kolom].unique()[:10])

    print("-"*60)

## 3.4 Penanganan Missing Value

In [ ]:
# ==========================================
# Penanganan Missing Value pada Fitur
# ==========================================

import numpy as np

# Membuat salinan dataset agar data asli tetap tersedia
df_imputed = df.copy()

# ------------------------------------------
# Mengubah nilai tidak valid menjadi NaN
# ------------------------------------------

# Nilai 999.99 pada ZS BB/TB dianggap sebagai
# nilai tidak valid / missing value implisit
jumlah_999 = (df_imputed["ZS BB/TB"] == 999.99).sum()

df_imputed["ZS BB/TB"] = df_imputed["ZS BB/TB"].replace(
    999.99,
    np.nan
)

print(
    f"Nilai 999.99 pada ZS BB/TB yang dikonversi menjadi NaN: "
    f"{jumlah_999}"
)

# ------------------------------------------
# Imputasi median pada variabel numerik
# ------------------------------------------

numerical_missing = [
    "TB Lahir",
    "LiLA",
    "Jml Vit A",
    "ZS BB/TB"
]

for kolom in numerical_missing:

    nilai_median = df_imputed[kolom].median()

    df_imputed[kolom] = (
        df_imputed[kolom]
        .fillna(nilai_median)
    )

    print(
        f"{kolom} diimputasi menggunakan median: "
        f"{nilai_median}"
    )

# ------------------------------------------
# Imputasi modus pada variabel kategorikal
# ------------------------------------------

categorical_missing = [
    "MBG",
    "Kelas Ibu Balita",
    "KPSP",
    "KIA"
]

for kolom in categorical_missing:

    nilai_modus = (
        df_imputed[kolom]
        .mode(dropna=True)[0]
    )

    df_imputed[kolom] = (
        df_imputed[kolom]
        .fillna(nilai_modus)
    )

    print(
        f"{kolom} diimputasi menggunakan modus: "
        f"{nilai_modus}"
    )

print(
    "\nPenanganan missing value pada seluruh fitur "
    "berhasil dilakukan."
)

# ------------------------------------------
# Verifikasi ZS BB/TB
# ------------------------------------------

print("\nVerifikasi ZS BB/TB:")
print(
    "Jumlah nilai 999.99 :",
    (df_imputed["ZS BB/TB"] == 999.99).sum()
)

print(
    "Jumlah missing value:",
    df_imputed["ZS BB/TB"].isna().sum()
)

print(
    "Rentang nilai:",
    df_imputed["ZS BB/TB"].min(),
    "sampai",
    df_imputed["ZS BB/TB"].max()
)

## 3.5 Validasi Hasil Penanganan Missing Value

In [ ]:
# ==========================================
# Validasi Hasil Penanganan Missing Value
# ==========================================

missing_after = pd.DataFrame({
    "Jumlah Missing": df_imputed.isnull().sum(),
    "Persentase Missing (%)": (
        df_imputed.isnull().sum() / len(df_imputed) * 100
    ).round(2)
})

missing_after = missing_after[
    missing_after["Jumlah Missing"] > 0
].sort_values(
    by="Persentase Missing (%)",
    ascending=False
)

print(f"Jumlah data setelah imputasi : {len(df_imputed)}")

if missing_after.empty:
    print("Tidak terdapat missing value pada dataset.")
else:
    display(missing_after)

In [ ]:
import matplotlib.pyplot as plt

# Variabel numerik setelah penanganan missing value
numeric_cols = [
    'TB Lahir',
    'LiLA',
    'Jml Vit A',
    'ZS BB/TB'
]

# Boxplot setelah penanganan missing value
plt.figure(figsize=(10, 6))

df[numeric_cols].boxplot()

plt.title('Boxplot Variabel Numerik Setelah Penanganan Missing Value')
plt.ylabel('Nilai')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3.6 Pemeriksaan Kategori Variabel Target (TB/U)

In [ ]:
# ==========================================
# Pemeriksaan Kategori Variabel Target TB/U
# ==========================================

distribusi_tbu_setelah_imputasi = (
    df_imputed["TB/U"]
    .value_counts(dropna=False)
    .reset_index()
)

distribusi_tbu_setelah_imputasi.columns = [
    "Kategori TB/U",
    "Jumlah Data"
]

display(distribusi_tbu_setelah_imputasi)

## 3.7 Penanganan Outlier pada Variabel TB/U

In [ ]:
# ==========================================
# Penanganan Outlier pada Variabel TB/U
# ==========================================

df_model = df_imputed[
    df_imputed["TB/U"] != "Outlier"
].copy()

## 3.8 Validasi Hasil Penanganan Outlier

In [ ]:
# ==========================================
# Validasi Hasil Penanganan Outlier
# ==========================================

print("=" * 55)
print("HASIL PENANGANAN OUTLIER")
print("=" * 55)

print("Jumlah data sebelum penanganan :", len(df_imputed))
print("Jumlah Outlier sebelum penanganan :", (df_imputed["TB/U"] == "Outlier").sum())
print("Jumlah data setelah penanganan :", len(df_model))
print("Jumlah Outlier setelah penanganan :", (df_model["TB/U"] == "Outlier").sum())

print("\nDistribusi kategori TB/U setelah penanganan:")
print(df_model["TB/U"].value_counts())

## 3.9 Transformasi Variabel Usia Saat Ukur

In [ ]:
# ============================================================
# Konversi Variabel Usia Saat Ukur Menjadi Usia dalam Bulan
# ============================================================

import re
import pandas as pd


# Membuat salinan dataset hasil imputasi
df_transformasi = df_imputed.copy()


def konversi_usia_ke_bulan(nilai):
    """
    Mengubah format:
    '2 Tahun - 8 Bulan - 1 Hari'

    menjadi usia dalam satuan bulan.
    """

    pola = (
        r"(\d+)\s*Tahun\s*-\s*"
        r"(\d+)\s*Bulan\s*-\s*"
        r"(\d+)\s*Hari"
    )

    hasil = re.search(
        pola,
        str(nilai)
    )

    if hasil is None:
        return None

    tahun = int(hasil.group(1))
    bulan = int(hasil.group(2))
    hari = int(hasil.group(3))

    usia_bulan = (
        (tahun * 12)
        + bulan
        + (hari / 30)
    )

    return usia_bulan


# ------------------------------------------------------------
# Mengubah Usia Saat Ukur menjadi bulan
# ------------------------------------------------------------

df_transformasi["Usia Saat Ukur"] = (
    df_transformasi["Usia Saat Ukur"]
    .apply(konversi_usia_ke_bulan)
)


# ------------------------------------------------------------
# Mengganti nama kolom
# ------------------------------------------------------------

df_transformasi = df_transformasi.rename(
    columns={
        "Usia Saat Ukur": "Usia (Bulan)"
    }
)


# ------------------------------------------------------------
# Membulatkan usia
# ------------------------------------------------------------

df_transformasi["Usia (Bulan)"] = (
    df_transformasi["Usia (Bulan)"]
    .round(2)
)


# ------------------------------------------------------------
# Mengatur posisi kolom
# ------------------------------------------------------------

urutan_kolom = list(
    df_transformasi.columns
)

urutan_kolom.remove(
    "Usia (Bulan)"
)

posisi_usia = (
    urutan_kolom.index("TB Lahir") + 1
)

urutan_kolom.insert(
    posisi_usia,
    "Usia (Bulan)"
)

df_transformasi = df_transformasi[
    urutan_kolom
]


# ------------------------------------------------------------
# Verifikasi
# ------------------------------------------------------------

print(
    "Konversi usia menjadi bulan "
    "berhasil dilakukan."
)

print(
    "\nTipe data Usia (Bulan):",
    df_transformasi["Usia (Bulan)"].dtype
)

print(
    "\nRentang usia:"
)

print(
    "Minimum:",
    df_transformasi["Usia (Bulan)"].min()
)

print(
    "Maximum:",
    df_transformasi["Usia (Bulan)"].max()
)

display(
    df_transformasi[
        ["Usia (Bulan)"]
    ].head(10)
)

## 3.10 Validasi Transformasi Usia Saat Ukur

In [ ]:
# ==========================================
# Validasi Variabel Usia (Bulan)
# ==========================================

print(
    "Tipe data Usia (Bulan) :",
    df_transformasi["Usia (Bulan)"].dtype
)

print(
    "Jumlah missing Usia (Bulan) :",
    df_transformasi["Usia (Bulan)"].isnull().sum()
)

print(
    "Rentang usia:",
    df_transformasi["Usia (Bulan)"].min(),
    "sampai",
    df_transformasi["Usia (Bulan)"].max(),
    "bulan"
)

display(
    df_transformasi[["Usia (Bulan)"]].head(10)
)

In [ ]:
# ============================================================
# VALIDASI FINAL Z-SCORE DATASET VS PERHITUNGAN WHO
# ============================================================

import pandas as pd

# Ambil beberapa contoh data yang lengkap
data_validasi = (
    df_transformasi[
        [
            "JK",
            "Usia (Bulan)",
            "Berat",
            "Tinggi",
            "ZS BB/U",
            "ZS BB/TB"
        ]
    ]
    .dropna()
    .head(10)
    .copy()
)

print("=" * 70)
print("DATA UNTUK VALIDASI Z-SCORE")
print("=" * 70)

display(data_validasi)

## 3.11 Transformasi Variabel Target

In [ ]:
# ==========================================
# Transformasi Variabel Target
# ==========================================

df_model = df_transformasi.copy()

# Menghapus kategori Outlier
df_model = (
    df_model[df_model["TB/U"] != "Outlier"]
    .reset_index(drop=True)
)

# Mengubah kategori TB/U menjadi target biner
mapping_target = {
    "Normal": "Normal",
    "Tinggi": "Normal",
    "Pendek": "Stunting",
    "Sangat Pendek": "Stunting"
}

df_model["Target"] = df_model["TB/U"].map(mapping_target)

print("Distribusi target setelah transformasi:")

display(
    df_model["Target"]
    .value_counts()
    .rename_axis("Kategori")
    .reset_index(name="Jumlah Data")
)

## 3.12 Encoding Variabel Kategorikal

In [ ]:
# ==========================================
# Encoding Variabel Kategorikal
# ==========================================

from sklearn.preprocessing import LabelEncoder

# Membuat salinan dataset
df_encoded = df_model.copy()

# Menentukan variabel kategorikal yang akan di-encoding
categorical_columns = [
    kolom
    for kolom in df_encoded.select_dtypes(include="object").columns
    if kolom != "Usia (Bulan)"
]

# Menyimpan encoder setiap variabel
label_encoders = {}

# Melakukan Label Encoding
for kolom in categorical_columns:
    encoder = LabelEncoder()

    df_encoded[kolom] = encoder.fit_transform(
        df_encoded[kolom]
    )

    label_encoders[kolom] = encoder


print("Variabel yang berhasil dilakukan Label Encoding:")

for kolom in categorical_columns:
    print(f"- {kolom}")


# Verifikasi usia
print(
    "\nTipe data Usia (Bulan):",
    df_encoded["Usia (Bulan)"].dtype
)

print(
    "Rentang Usia (Bulan):",
    df_encoded["Usia (Bulan)"].min(),
    "sampai",
    df_encoded["Usia (Bulan)"].max()
)

In [ ]:
# ============================================================
# Cek Mapping Encoding Variabel Naik Berat Badan
# ============================================================

print("Nilai asli Naik Berat Badan:")
print(df["Naik Berat Badan"].value_counts(dropna=False))

print("\nNilai setelah encoding:")
print(df_encoded["Naik Berat Badan"].value_counts(dropna=False))

## 3.13 Verifikasi Hasil Encoding

In [ ]:
# ==========================================
# Pemeriksaan Tipe Data Setelah Encoding
# ==========================================

display(df_encoded.dtypes)

In [ ]:
# ============================================================
# CEK MAPPING HASIL LABEL ENCODING
# ============================================================

print("=" * 60)
print("MAPPING ENCODING VARIABEL")
print("=" * 60)

for kolom in [
    "JK",
    "Cara Ukur",
    "Naik Berat Badan",
]:
    encoder = label_encoders[kolom]

    print(f"\n{kolom}:")

    for kode, kelas in enumerate(
        encoder.classes_
    ):
        print(
            f"{kode} = {kelas}"
        )

In [ ]:
# ============================================================
# CEK DATA ASLI UNTUK PERBANDINGAN Z-SCORE
# ============================================================

kolom_cek = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "Cara Ukur",
    "ZS BB/U",
    "ZS BB/TB",
    "BB/U",
    "Naik Berat Badan",
]

# Ambil beberapa sampel
display(
    df_model[
        kolom_cek
    ].head(10)
)

In [ ]:
# ============================================================
# ANALISIS NAIK BERAT BADAN TERHADAP TARGET
# SEBELUM RANDOM UNDER SAMPLING
# ============================================================

import pandas as pd

print("=" * 70)
print("HUBUNGAN NAIK BERAT BADAN DENGAN TARGET - DATA ASLI")
print("=" * 70)

# Mapping hasil LabelEncoder
mapping_naik_bb = {
    0: "B (Baru)",
    1: "N (Naik)",
    2: "O (Tidak hadir bulan lalu)",
    3: "T (Tidak naik)"
}

# Crosstab jumlah data
tabel = pd.crosstab(
    df_encoded["Naik Berat Badan"],
    df_encoded["Target"],
    margins=True
)

print("\nJUMLAH DATA:")
display(tabel)


# ============================================================
# PERSENTASE TARGET PADA SETIAP KATEGORI
# ============================================================

persentase = pd.crosstab(
    df_encoded["Naik Berat Badan"],
    df_encoded["Target"],
    normalize="index"
) * 100

persentase = persentase.rename(
    index=mapping_naik_bb
)

persentase = persentase.rename(
    columns={
        0: "Normal (%)",
        1: "Stunting (%)"
    }
)

print("\nPERSENTASE TARGET PADA SETIAP KATEGORI:")
display(persentase.round(2))


# ============================================================
# KHUSUS MEMBANDINGKAN NAIK DAN TIDAK NAIK
# ============================================================

print("\n" + "=" * 70)
print("PERBANDINGAN NAIK (N) DAN TIDAK NAIK (T)")
print("=" * 70)

for kode, nama in [
    (1, "N / Naik"),
    (3, "T / Tidak naik")
]:
    subset = df_encoded[
        df_encoded["Naik Berat Badan"] == kode
    ]

    print(f"\n{nama}")
    print("Jumlah data :", len(subset))

    print("Distribusi target:")
    print(
        subset["Target"]
        .value_counts()
        .sort_index()
    )

    print("Persentase:")
    print(
        (
            subset["Target"]
            .value_counts(normalize=True)
            .sort_index() * 100
        ).round(2)
    )

In [ ]:
# ============================================================
# ANALISIS TARGET BERDASARKAN NAIK BB DAN STATUS TB/U
# ============================================================

tabel_kombinasi = pd.crosstab(
    [
        df_encoded["Naik Berat Badan"],
        df_encoded["TB/U"]
    ],
    df_encoded["Target"],
    margins=True
)

print("HUBUNGAN NAIK BERAT BADAN + TB/U DENGAN TARGET")
display(tabel_kombinasi)

In [ ]:
# ============================================================
# CEK DATA ASLI YANG MIRIP DENGAN INPUT WEB
# ============================================================

import numpy as np

# Input web
usia_input = 24
berat_input = 11.5
tinggi_input = 85
zs_bbu_input = 0.02
zs_bbtb_input = 0.28

# N = Naik
naik_bb_input = 1

# BB/U Normal
bbu_input = 1

data_cek = df_encoded.copy()

# Ambil data dengan status Naik dan BB/U yang sama
data_cek = data_cek[
    (data_cek["Naik Berat Badan"] == naik_bb_input) &
    (data_cek["BB/U"] == bbu_input)
].copy()

# Hitung jarak terhadap input web
data_cek["Jarak_Input"] = np.sqrt(
    ((data_cek["Usia (Bulan)"] - usia_input) / 60) ** 2 +
    ((data_cek["Berat"] - berat_input) / 20) ** 2 +
    ((data_cek["Tinggi"] - tinggi_input) / 120) ** 2 +
    ((data_cek["ZS BB/U"] - zs_bbu_input) / 6) ** 2 +
    ((data_cek["ZS BB/TB"] - zs_bbtb_input) / 6) ** 2
)

kolom_tampil = [
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "ZS BB/U",
    "ZS TB/U",
    "ZS BB/TB",
    "BB/U",
    "TB/U",
    "Naik Berat Badan",
    "Target",
    "Jarak_Input"
]

hasil_mirip = (
    data_cek[kolom_tampil]
    .sort_values("Jarak_Input")
    .head(30)
)

print("=" * 70)
print("30 DATA ASLI PALING MIRIP DENGAN INPUT WEB")
print("=" * 70)

display(hasil_mirip)

print("\nDistribusi Target 30 Data Terdekat:")
print(hasil_mirip["Target"].value_counts().sort_index())

print("\nKeterangan:")
print("0 = Normal")
print("1 = Stunting")

## 3.14 Pembentukan Fitur dan Target

In [ ]:
# ============================================================
# Pembentukan Fitur dan Target
# ============================================================

# Menyalin dataset hasil encoding
df_model = df_encoded.copy()

# --------------------------------------------
# Menentukan variabel yang tidak digunakan
# sebagai fitur model
# --------------------------------------------
fitur_dihapus = [
    "Target",
    "TB/U",
    "ZS TB/U"
]

# --------------------------------------------
# Variabel independen (X)
# --------------------------------------------
X = df_model.drop(
    columns=fitur_dihapus
)

# --------------------------------------------
# Variabel dependen / target (y)
# --------------------------------------------
y = df_model["Target"].copy()

# --------------------------------------------
# Menampilkan hasil
# --------------------------------------------
print("Jumlah data       :", X.shape[0])
print("Jumlah fitur awal :", X.shape[1])

print("\nDaftar fitur yang akan menjadi kandidat seleksi Mutual Information:")
for i, fitur in enumerate(X.columns, start=1):
    print(f"{i}. {fitur}")

print("\nDistribusi Target:")
print(y.value_counts().sort_index())

print("\nKeterangan Target:")
print("0 = Normal")
print("1 = Stunting")

## 3.15 Pembagian Data Training dan Testing

In [ ]:
# ============================================================
# Pembagian Data Training dan Testing
# ============================================================

from sklearn.model_selection import train_test_split

# Membagi dataset dengan rasio 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Menampilkan ukuran dataset
print("Ukuran data training :", X_train.shape)
print("Ukuran data testing  :", X_test.shape)

# Menampilkan distribusi target
print("\nDistribusi target training:")
print(y_train.value_counts().sort_index())

print("\nDistribusi target testing:")
print(y_test.value_counts().sort_index())

# Menampilkan persentase kelas
print("\nPersentase target training:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nPersentase target testing:")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))

## 3.16 Penetapan Fitur Penelitian

In [ ]:
# ============================================================
# Penetapan Fitur Berdasarkan Penelitian Terdahulu
# ============================================================

# Menentukan fitur yang digunakan dalam proses pemodelan
fitur_terpilih = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "LiLA"
]

# ============================================================
# Membentuk data training dan testing dengan 5 fitur
# ============================================================

X_train_selected = X_train[fitur_terpilih].copy()
X_test_selected = X_test[fitur_terpilih].copy()

# ============================================================
# Menampilkan fitur yang digunakan
# ============================================================

print("============================================================")
print("PENETAPAN FITUR")
print("============================================================")

print("\nFitur yang digunakan dalam pemodelan:")

for i, fitur in enumerate(fitur_terpilih, start=1):
    print(f"{i}. {fitur}")

print("\nJumlah fitur yang digunakan :", len(fitur_terpilih))

# ============================================================
# Menampilkan ukuran data setelah penetapan fitur
# ============================================================

print("\nUkuran data setelah penetapan fitur:")
print("Data training :", X_train_selected.shape)
print("Data testing  :", X_test_selected.shape)

# ============================================================
# Verifikasi nama fitur
# ============================================================

print("\nKolom fitur training:")
print(X_train_selected.columns.tolist())

print("\nKolom fitur testing:")
print(X_test_selected.columns.tolist())

## 3.17 Verifikasi Fitur Terpilih

In [ ]:
# ============================================================
# Verifikasi Fitur Terpilih
# ============================================================

print("=" * 60)
print("VERIFIKASI FITUR TERPILIH")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verifikasi jumlah fitur
# ------------------------------------------------------------

print("\nJumlah fitur training :", X_train_selected.shape[1])
print("Jumlah fitur testing  :", X_test_selected.shape[1])

# ------------------------------------------------------------
# 2. Verifikasi nama fitur
# ------------------------------------------------------------

print("\nFitur yang digunakan:")
for i, fitur in enumerate(X_train_selected.columns, start=1):
    print(f"{i}. {fitur}")

# ------------------------------------------------------------
# 3. Verifikasi kesamaan fitur training dan testing
# ------------------------------------------------------------

print("\nFitur training dan testing sama :",
      list(X_train_selected.columns) == list(X_test_selected.columns))

# ------------------------------------------------------------
# 4. Verifikasi ukuran data
# ------------------------------------------------------------

print("\nUkuran data training :", X_train_selected.shape)
print("Ukuran data testing  :", X_test_selected.shape)

# ------------------------------------------------------------
# 5. Verifikasi apakah masih terdapat fitur yang tidak digunakan
# ------------------------------------------------------------

fitur_tidak_digunakan = [
    fitur for fitur in X.columns
    if fitur not in fitur_terpilih
]

print("\nJumlah fitur yang tidak digunakan :", len(fitur_tidak_digunakan))
print("Fitur yang tidak digunakan :")
print(fitur_tidak_digunakan)

## 3.18 Verifikasi Duplikasi Data

In [ ]:
# ============================================================
# VERIFIKASI DUPLIKASI DATA
# ============================================================

import pandas as pd
import numpy as np

print("======================================================================")
print("VERIFIKASI DUPLIKASI DATA")
print("======================================================================")

# ------------------------------------------------------------
# 1. Jumlah seluruh data
# ------------------------------------------------------------

jumlah_data = len(df)

# ------------------------------------------------------------
# 2. Menentukan baris yang termasuk kelompok duplikat
# ------------------------------------------------------------

duplikat_semua = df.duplicated(keep=False)

jumlah_baris_kelompok_duplikat = duplikat_semua.sum()

# ------------------------------------------------------------
# 3. Jumlah data duplikat setelah kemunculan pertama
# ------------------------------------------------------------

jumlah_data_duplikat = df.duplicated(keep="first").sum()

# ------------------------------------------------------------
# 4. Jumlah kelompok kombinasi data yang duplikat
# ------------------------------------------------------------

jumlah_kelompok_duplikat = (
    df.value_counts()
      .loc[lambda x: x > 1]
      .shape[0]
)

print(f"Jumlah seluruh data : {jumlah_data}")
print(
    f"Jumlah baris yang termasuk kelompok duplikat : "
    f"{jumlah_baris_kelompok_duplikat}"
)
print(
    f"Jumlah data duplikat setelah kemunculan pertama : "
    f"{jumlah_data_duplikat}"
)
print(
    f"Jumlah kelompok kombinasi fitur yang duplikat : "
    f"{jumlah_kelompok_duplikat}"
)


# ============================================================
# VERIFIKASI DATA CONTOH KNN
# ============================================================

print()
print("======================================================================")
print("VERIFIKASI DATA CONTOH KNN")
print("======================================================================")

# ------------------------------------------------------------
# 5. Index data testing yang digunakan sebagai contoh
# ------------------------------------------------------------

index_contoh_knn = 2

# ------------------------------------------------------------
# 6. Mengambil satu data testing
# ------------------------------------------------------------

data_contoh = X_test.iloc[index_contoh_knn]

print()
print("Index data testing :", index_contoh_knn)

print()
print("Data testing yang digunakan:")
print(data_contoh)


# ------------------------------------------------------------
# 7. Mencari data training yang memiliki nilai fitur sama
# ------------------------------------------------------------

data_sama = X_train.eq(data_contoh).all(axis=1)

# Mengambil indeks data training yang sama
indeks_sama = X_train.index[data_sama].tolist()


# ------------------------------------------------------------
# 8. Menampilkan hasil verifikasi
# ------------------------------------------------------------

print()
print("Jumlah data training dengan nilai fitur yang sama :",
      len(indeks_sama))

if len(indeks_sama) > 0:

    print("Indeks data training yang sama :")
    print(indeks_sama)

else:

    print("Tidak ditemukan data training dengan nilai fitur yang sama.")


# ============================================================
# VERIFIKASI KESESUAIAN FITUR TRAINING DAN TESTING
# ============================================================

print()
print("======================================================================")
print("VERIFIKASI FITUR TRAINING DAN TESTING")
print("======================================================================")

print("Jumlah fitur training :", X_train.shape[1])
print("Jumlah fitur testing  :", X_test.shape[1])

fitur_sama = list(X_train.columns) == list(X_test.columns)

print("Fitur training dan testing sama :", fitur_sama)

print()
print("Fitur yang digunakan:")

for i, fitur in enumerate(X_train.columns, start=1):
    print(f"{i}. {fitur}")

## 3.19 Verifikasi Duplikasi Berdasarkan Target

In [ ]:
# ============================================================
# 46. VERIFIKASI DUPLIKASI BERDASARKAN TARGET
# ============================================================

fitur_cek = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "LiLA"
]

# ------------------------------------------------------------
# Kelompok berdasarkan 5 fitur
# ------------------------------------------------------------

kelompok = (
    df_encoded
    .groupby(fitur_cek, dropna=False)["Target"]
    .agg(
        jumlah_data="size",
        jumlah_kelas="nunique"
    )
    .reset_index()
)

# ------------------------------------------------------------
# Kelompok fitur yang memiliki lebih dari satu target
# ------------------------------------------------------------

konflik_target = kelompok[
    kelompok["jumlah_kelas"] > 1
]

# ------------------------------------------------------------
# Kelompok fitur yang hanya memiliki satu target
# ------------------------------------------------------------

duplikat_target_konsisten = kelompok[
    (kelompok["jumlah_data"] > 1) &
    (kelompok["jumlah_kelas"] == 1)
]

# ------------------------------------------------------------
# Menampilkan hasil ringkas
# ------------------------------------------------------------

print("=" * 70)
print("VERIFIKASI DUPLIKASI BERDASARKAN TARGET")
print("=" * 70)

print(
    "\nJumlah kombinasi fitur :",
    len(kelompok)
)

print(
    "Kombinasi fitur dengan data berulang :",
    (kelompok["jumlah_data"] > 1).sum()
)

print(
    "Duplikasi dengan target konsisten :",
    len(duplikat_target_konsisten)
)

print(
    "Duplikasi dengan target berbeda :",
    len(konflik_target)
)

print("\nJumlah baris pada kelompok target berbeda :")

print(
    df_encoded.merge(
        konflik_target[fitur_cek],
        on=fitur_cek,
        how="inner"
    ).shape[0]
)

## 3.20 Verifikasi Duplikasi antara Training dan Testing

In [ ]:
# ============================================================
# 46. VERIFIKASI DUPLIKASI ANTARA TRAINING DAN TESTING
# ============================================================

fitur_cek = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "LiLA"
]

# ------------------------------------------------------------
# Membuat kombinasi fitur training
# ------------------------------------------------------------

fitur_training = set(
    map(
        tuple,
        X_train_selected[fitur_cek].to_numpy()
    )
)

# ------------------------------------------------------------
# Mengecek kombinasi fitur testing yang juga terdapat
# pada data training
# ------------------------------------------------------------

fitur_testing = list(
    map(
        tuple,
        X_test_selected[fitur_cek].to_numpy()
    )
)

overlap = [
    data
    for data in fitur_testing
    if data in fitur_training
]

# ------------------------------------------------------------
# Menghitung jumlah dan persentase
# ------------------------------------------------------------

jumlah_overlap = len(overlap)

jumlah_testing = len(X_test_selected)

persentase_overlap = (
    jumlah_overlap /
    jumlah_testing *
    100
)

# ------------------------------------------------------------
# Menampilkan hasil
# ------------------------------------------------------------

print("=" * 70)
print("VERIFIKASI DUPLIKASI TRAINING DAN TESTING")
print("=" * 70)

print(
    "\nJumlah data training :",
    len(X_train_selected)
)

print(
    "Jumlah data testing  :",
    jumlah_testing
)

print(
    "\nData testing yang memiliki kombinasi fitur "
    "yang juga terdapat pada training :",
    jumlah_overlap
)

print(
    "Persentase overlap :",
    round(persentase_overlap, 2),
    "%"
)

print(
    "\nTerdapat overlap training-testing :",
    jumlah_overlap > 0
)

## 3.21 Pemeriksaan Kolom Identitas Dataset

In [ ]:
# ============================================================
# 47. PEMERIKSAAN KOLOM IDENTITAS DATASET
# ============================================================

print("=" * 70)
print("DAFTAR KOLOM DATASET")
print("=" * 70)

print("\nJumlah kolom :", len(df.columns))

for i, kolom in enumerate(df.columns, start=1):
    print(f"{i}. {kolom}")

## 3.22 Penerapan Random Under Sampling

In [ ]:
# ============================================================
# Penerapan Random Under Sampling pada Data Training
# ============================================================

from imblearn.under_sampling import RandomUnderSampler

# ------------------------------------------------------------
# Menentukan skenario rasio RUS
# ------------------------------------------------------------

rasio_rus = {
    "RUS 3:1": 1/3,
    "RUS 2.5:1": 1/2.5,
    "RUS 2:1": 1/2
}

print("=" * 60)
print("RANDOM UNDER SAMPLING")
print("=" * 60)

# ------------------------------------------------------------
# Distribusi kelas sebelum RUS
# ------------------------------------------------------------

print("\nDistribusi kelas sebelum RUS:")

print(
    y_train
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

print("\nUkuran data sebelum RUS:")
print(X_train_selected.shape)

# ------------------------------------------------------------
# Menyimpan hasil setiap skenario
# ------------------------------------------------------------

hasil_rus = {}

# ------------------------------------------------------------
# Menerapkan setiap skenario RUS
# ------------------------------------------------------------

for nama_skenario, rasio in rasio_rus.items():

    rus = RandomUnderSampler(
        sampling_strategy=rasio,
        random_state=42
    )

    X_rus, y_rus = rus.fit_resample(
        X_train_selected,
        y_train
    )

    hasil_rus[nama_skenario] = {
        "X": X_rus,
        "y": y_rus
    }

    print("\n" + "-" * 60)
    print(nama_skenario)

    print("\nDistribusi kelas setelah RUS:")

    print(
        y_rus
        .value_counts()
        .sort_index()
        .rename(index={
            0: "Normal",
            1: "Stunting"
        })
    )

    print("\nUkuran data setelah RUS:")
    print(X_rus.shape)

# ------------------------------------------------------------
# Data testing tetap tidak dilakukan RUS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATA TESTING")
print("=" * 60)

print("Ukuran data testing :", X_test_selected.shape)

print("\nDistribusi target testing:")

print(
    y_test
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

## 3.23 Verifikasi Hasil Random Under Sampling

In [ ]:
# ============================================================
# Verifikasi Hasil Random Under Sampling
# ============================================================

print("=" * 70)
print("VERIFIKASI HASIL RANDOM UNDER SAMPLING")
print("=" * 70)

# ------------------------------------------------------------
# Distribusi sebelum RUS
# ------------------------------------------------------------

distribusi_sebelum = (
    y_train
    .value_counts()
    .sort_index()
)

print("\nDistribusi sebelum RUS:")
print(
    distribusi_sebelum
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Persentase sebelum RUS
# ------------------------------------------------------------

persentase_sebelum = (
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nPersentase sebelum RUS:")
print(
    persentase_sebelum
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Verifikasi setiap skenario RUS
# ------------------------------------------------------------

for nama_skenario, hasil in hasil_rus.items():

    y_rus = hasil["y"]

    distribusi = (
        y_rus
        .value_counts()
        .sort_index()
    )

    persentase = (
        y_rus
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    print("\n" + "-" * 70)
    print(nama_skenario)

    print("\nDistribusi kelas:")
    print(
        distribusi
        .rename(index={
            0: "Normal",
            1: "Stunting"
        })
    )

    print("\nPersentase kelas:")
    print(
        persentase
        .rename(index={
            0: "Normal",
            1: "Stunting"
        })
    )

    print("\nJumlah data :", len(y_rus))

    # Verifikasi jumlah kelas minoritas
    print(
        "Jumlah Stunting tetap :",
        distribusi.get(1, 0)
    )

# ------------------------------------------------------------
# Verifikasi data testing
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VERIFIKASI DATA TESTING")
print("=" * 70)

print("\nUkuran data testing :", X_test_selected.shape)

print("\nDistribusi target testing:")
print(
    y_test
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

## 3.24 Visualisasi Hasil Random Under Sampling

In [ ]:
# ============================================================
# Visualisasi Hasil Random Under Sampling
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Data distribusi kelas
# ------------------------------------------------------------

skenario = [
    "Sebelum RUS",
    "RUS 3:1",
    "RUS 2.5:1",
    "RUS 2:1"
]

normal = [
    y_train.value_counts().get(0, 0),
    hasil_rus["RUS 3:1"]["y"].value_counts().get(0, 0),
    hasil_rus["RUS 2.5:1"]["y"].value_counts().get(0, 0),
    hasil_rus["RUS 2:1"]["y"].value_counts().get(0, 0)
]

stunting = [
    y_train.value_counts().get(1, 0),
    hasil_rus["RUS 3:1"]["y"].value_counts().get(1, 0),
    hasil_rus["RUS 2.5:1"]["y"].value_counts().get(1, 0),
    hasil_rus["RUS 2:1"]["y"].value_counts().get(1, 0)
]

# ------------------------------------------------------------
# Membuat posisi batang
# ------------------------------------------------------------

x = np.arange(len(skenario))
width = 0.35

# ------------------------------------------------------------
# Membuat grafik
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

bars_normal = plt.bar(
    x - width / 2,
    normal,
    width,
    label="Normal"
)

bars_stunting = plt.bar(
    x + width / 2,
    stunting,
    width,
    label="Stunting"
)

# ------------------------------------------------------------
# Menambahkan nilai pada setiap batang
# ------------------------------------------------------------

for bars in [bars_normal, bars_stunting]:
    for bar in bars:
        tinggi = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            tinggi,
            f"{int(tinggi):,}".replace(",", "."),
            ha="center",
            va="bottom"
        )

# ------------------------------------------------------------
# Pengaturan grafik
# ------------------------------------------------------------

plt.title("Distribusi Kelas Sebelum dan Setelah Random Under Sampling")
plt.xlabel("Skenario RUS")
plt.ylabel("Jumlah Data")
plt.xticks(x, skenario)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 3.25 Penerapan Regular SMOTE

In [ ]:
# ============================================================
# Penerapan Regular SMOTE
# ============================================================

from imblearn.over_sampling import SMOTE

print("=" * 60)
print("REGULAR SMOTE")
print("=" * 60)

# ------------------------------------------------------------
# Parameter SMOTE
# ------------------------------------------------------------

k_neighbors = 5
random_state = 42

print("\nParameter SMOTE:")
print("Metode        : Regular SMOTE")
print("k_neighbors   :", k_neighbors)
print("random_state  :", random_state)

# ------------------------------------------------------------
# Menyimpan hasil SMOTE setiap skenario RUS
# ------------------------------------------------------------

hasil_smote = {}

# ------------------------------------------------------------
# Menerapkan SMOTE pada setiap hasil RUS
# ------------------------------------------------------------

for nama_skenario, hasil in hasil_rus.items():

    X_rus = hasil["X"]
    y_rus = hasil["y"]

    smote = SMOTE(
        sampling_strategy="auto",
        k_neighbors=k_neighbors,
        random_state=random_state
    )

    X_smote, y_smote = smote.fit_resample(
        X_rus,
        y_rus
    )

    hasil_smote[nama_skenario] = {
        "X": X_smote,
        "y": y_smote
    }

    print("\n" + "-" * 60)
    print(nama_skenario)

    print("\nDistribusi kelas setelah SMOTE:")
    print(
        y_smote
        .value_counts()
        .sort_index()
        .rename(index={
            0: "Normal",
            1: "Stunting"
        })
    )

    print("\nUkuran data setelah SMOTE:")
    print(X_smote.shape)

# ------------------------------------------------------------
# Data testing tidak dilakukan SMOTE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATA TESTING")
print("=" * 60)

print("\nUkuran data testing :", X_test_selected.shape)

print("\nDistribusi target testing:")
print(
    y_test
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

## 3.26 Verifikasi Hasil Regular SMOTE

In [ ]:
# ============================================================
# Verifikasi Hasil Regular SMOTE
# ============================================================

print("=" * 70)
print("VERIFIKASI HASIL REGULAR SMOTE")
print("=" * 70)

for nama_skenario, hasil in hasil_smote.items():

    y_smote = hasil["y"]

    # Distribusi kelas
    distribusi = (
        y_smote
        .value_counts()
        .sort_index()
    )

    # Persentase kelas
    persentase = (
        y_smote
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    print("\n" + "-" * 70)
    print(nama_skenario)

    print("\nDistribusi kelas setelah SMOTE:")
    print(
        distribusi
        .rename(index={
            0: "Normal",
            1: "Stunting"
        })
    )

    print("\nPersentase kelas setelah SMOTE:")
    print(
        persentase
        .rename(index={
            0: "Normal",
            1: "Stunting"
        })
    )

    print("\nJumlah data :", len(y_smote))

    # Verifikasi keseimbangan kelas
    jumlah_normal = distribusi.get(0, 0)
    jumlah_stunting = distribusi.get(1, 0)

    print(
        "Kelas seimbang :",
        jumlah_normal == jumlah_stunting
    )

    # Verifikasi jumlah fitur
    print(
        "Jumlah fitur   :",
        hasil["X"].shape[1]
    )

# ============================================================
# Verifikasi data testing
# ============================================================

print("\n" + "=" * 70)
print("VERIFIKASI DATA TESTING")
print("=" * 70)

print("\nUkuran data testing :", X_test_selected.shape)

print("\nDistribusi target testing:")
print(
    y_test
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

## 3.27 Visualisasi Distribusi Hasil Regular SMOTE

In [ ]:
# ============================================================
# Visualisasi Distribusi Hasil Regular SMOTE
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Menentukan skenario
# ------------------------------------------------------------

skenario = [
    "RUS 3:1 + SMOTE",
    "RUS 2.5:1 + SMOTE",
    "RUS 2:1 + SMOTE"
]

# ------------------------------------------------------------
# Mengambil jumlah masing-masing kelas
# ------------------------------------------------------------

normal = [
    hasil_smote["RUS 3:1"]["y"].value_counts().get(0, 0),
    hasil_smote["RUS 2.5:1"]["y"].value_counts().get(0, 0),
    hasil_smote["RUS 2:1"]["y"].value_counts().get(0, 0)
]

stunting = [
    hasil_smote["RUS 3:1"]["y"].value_counts().get(1, 0),
    hasil_smote["RUS 2.5:1"]["y"].value_counts().get(1, 0),
    hasil_smote["RUS 2:1"]["y"].value_counts().get(1, 0)
]

# ------------------------------------------------------------
# Posisi batang
# ------------------------------------------------------------

x = np.arange(len(skenario))
width = 0.35

# ------------------------------------------------------------
# Membuat grafik
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

bars_normal = plt.bar(
    x - width / 2,
    normal,
    width,
    label="Normal"
)

bars_stunting = plt.bar(
    x + width / 2,
    stunting,
    width,
    label="Stunting"
)

# ------------------------------------------------------------
# Menambahkan nilai pada batang
# ------------------------------------------------------------

for bars in [bars_normal, bars_stunting]:
    for bar in bars:
        tinggi = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            tinggi,
            f"{int(tinggi):,}".replace(",", "."),
            ha="center",
            va="bottom"
        )

# ------------------------------------------------------------
# Pengaturan grafik
# ------------------------------------------------------------

plt.title("Distribusi Kelas Setelah Random Under Sampling dan Regular SMOTE")
plt.xlabel("Skenario Penyeimbangan Data")
plt.ylabel("Jumlah Data")
plt.xticks(x, skenario)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 3.28 Penentuan Skenario RUS + Regular SMOTE

In [ ]:
# ============================================================
# Penentuan Skenario RUS + Regular SMOTE
# Menggunakan Stratified 5-Fold Cross Validation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE


# ============================================================
# 1. SKENARIO RANDOM UNDER SAMPLING
# ============================================================

skenario_rus_cv = {
    "RUS 3:1": 1/3,
    "RUS 2.5:1": 1/2.5,
    "RUS 2:1": 1/2
}


# ============================================================
# 2. STRATIFIED 5-FOLD CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 3. PENYIMPANAN HASIL
# ============================================================

hasil_cv = []


# ============================================================
# 4. EVALUASI SETIAP SKENARIO RUS + SMOTE
# ============================================================

for nama_rus, rasio_rus in skenario_rus_cv.items():

    skor_rf = []
    skor_knn = []

    print("\n" + "=" * 70)
    print(nama_rus)
    print("=" * 70)

    # --------------------------------------------------------
    # Stratified 5-Fold
    # --------------------------------------------------------

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train_selected, y_train),
        start=1
    ):

        # ----------------------------------------------------
        # Membagi training fold dan validation fold
        # ----------------------------------------------------

        X_fold_train = X_train_selected.iloc[
            train_idx
        ].copy()

        X_fold_val = X_train_selected.iloc[
            val_idx
        ].copy()

        y_fold_train = y_train.iloc[
            train_idx
        ].copy()

        y_fold_val = y_train.iloc[
            val_idx
        ].copy()


        # ----------------------------------------------------
        # Random Under Sampling
        # HANYA pada training fold
        # ----------------------------------------------------

        rus = RandomUnderSampler(
            sampling_strategy=rasio_rus,
            random_state=42
        )

        X_fold_rus, y_fold_rus = rus.fit_resample(
            X_fold_train,
            y_fold_train
        )


        # ----------------------------------------------------
        # Regular SMOTE
        # HANYA pada hasil training fold
        # ----------------------------------------------------

        smote = SMOTE(
            sampling_strategy="auto",
            k_neighbors=5,
            random_state=42
        )

        X_fold_smote, y_fold_smote = smote.fit_resample(
            X_fold_rus,
            y_fold_rus
        )


        # ====================================================
        # RANDOM FOREST
        # ====================================================

        rf = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(
            X_fold_smote,
            y_fold_smote
        )

        pred_rf = rf.predict(
            X_fold_val
        )

        f1_rf = f1_score(
            y_fold_val,
            pred_rf,
            average="macro"
        )

        skor_rf.append(f1_rf)


        # ====================================================
        # K-NEAREST NEIGHBOR
        # ====================================================

        scaler = StandardScaler()

        X_knn_train = scaler.fit_transform(
            X_fold_smote
        )

        X_knn_val = scaler.transform(
            X_fold_val
        )

        knn = KNeighborsClassifier(
            n_neighbors=5
        )

        knn.fit(
            X_knn_train,
            y_fold_smote
        )

        pred_knn = knn.predict(
            X_knn_val
        )

        f1_knn = f1_score(
            y_fold_val,
            pred_knn,
            average="macro"
        )

        skor_knn.append(f1_knn)


        # ----------------------------------------------------
        # Menampilkan hasil setiap fold
        # ----------------------------------------------------

        print(
            f"Fold {fold} | "
            f"Macro F1 RF = {f1_rf:.4f} | "
            f"Macro F1 KNN = {f1_knn:.4f}"
        )


    # ========================================================
    # 5. RATA-RATA HASIL 5 FOLD
    # ========================================================

    mean_rf = np.mean(skor_rf)
    mean_knn = np.mean(skor_knn)

    mean_gabungan = (
        mean_rf + mean_knn
    ) / 2


    # ========================================================
    # 6. MENYIMPAN HASIL
    # ========================================================

    hasil_cv.append({
        "Skenario RUS": nama_rus,
        "Macro F1 RF": mean_rf,
        "Macro F1 KNN": mean_knn,
        "Rata-rata Macro F1": mean_gabungan
    })


# ============================================================
# 7. MEMBENTUK TABEL HASIL CV
# ============================================================

hasil_cv_df = pd.DataFrame(
    hasil_cv
)


# ============================================================
# 8. MENENTUKAN SKENARIO TERBAIK
# ============================================================

index_terbaik = hasil_cv_df[
    "Rata-rata Macro F1"
].idxmax()

skenario_rus_terbaik = hasil_cv_df.loc[
    index_terbaik,
    "Skenario RUS"
]


# ============================================================
# 9. MENAMPILKAN HASIL
# ============================================================

print("\n" + "=" * 70)
print("HASIL STRATIFIED 5-FOLD CROSS VALIDATION")
print("=" * 70)

display(
    hasil_cv_df.round(4)
)

print("\nSkenario RUS + Regular SMOTE terbaik :",
      skenario_rus_terbaik)

print(
    "Macro F1 Random Forest :",
    round(
        hasil_cv_df.loc[
            index_terbaik,
            "Macro F1 RF"
        ],
        4
    )
)

print(
    "Macro F1 KNN :",
    round(
        hasil_cv_df.loc[
            index_terbaik,
            "Macro F1 KNN"
        ],
        4
    )
)

print(
    "Rata-rata Macro F1 :",
    round(
        hasil_cv_df.loc[
            index_terbaik,
            "Rata-rata Macro F1"
        ],
        4
    )
)

# 4. Modelling

## 4.1 Pemodelan Random Forest

### 4.1.1 Persiapan Data Random Forest

In [ ]:
# ============================================================
# 35. PERSIAPAN DATA RANDOM FOREST
# Menggunakan Skenario RUS 3:1 + Regular SMOTE
# ============================================================

# ------------------------------------------------------------
# Menentukan skenario penyeimbangan yang terpilih
# ------------------------------------------------------------

skenario_terpilih = "RUS 3:1"

# ------------------------------------------------------------
# Mengambil data hasil RUS 3:1 + Regular SMOTE
# ------------------------------------------------------------

X_train_rf = hasil_smote[skenario_terpilih]["X"].copy()
y_train_rf = hasil_smote[skenario_terpilih]["y"].copy()

# ------------------------------------------------------------
# Data testing tetap menggunakan data asli
# ------------------------------------------------------------

X_test_rf = X_test_selected.copy()
y_test_rf = y_test.copy()

# ------------------------------------------------------------
# Menampilkan informasi data training
# ------------------------------------------------------------

print("=" * 70)
print("PERSIAPAN DATA RANDOM FOREST")
print("=" * 70)

print("\nSkenario penyeimbangan terpilih:")
print("RUS 3:1 + Regular SMOTE")

print("\nFitur yang digunakan:")
for i, fitur in enumerate(X_train_rf.columns, start=1):
    print(f"{i}. {fitur}")

print("\nJumlah fitur :", X_train_rf.shape[1])

print("\nUkuran data training setelah RUS + SMOTE:")
print(X_train_rf.shape)

print("\nDistribusi target training:")
print(
    y_train_rf
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Menampilkan informasi data testing
# ------------------------------------------------------------

print("\nUkuran data testing:")
print(X_test_rf.shape)

print("\nDistribusi target testing:")
print(
    y_test_rf
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Verifikasi kesesuaian fitur training dan testing
# ------------------------------------------------------------

print("\nVerifikasi fitur:")
print("Jumlah fitur training :", X_train_rf.shape[1])
print("Jumlah fitur testing  :", X_test_rf.shape[1])

print(
    "Fitur training dan testing sama :",
    list(X_train_rf.columns) == list(X_test_rf.columns)
)

### 4.1.2 Penentuan Hyperparameter Random Forest

In [ ]:
# ============================================================
# 36. PENENTUAN HYPERPARAMETER RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# ------------------------------------------------------------
# Model dasar Random Forest
# ------------------------------------------------------------

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# Kandidat hyperparameter
# ------------------------------------------------------------

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "criterion": ["gini"]
}

# ------------------------------------------------------------
# Stratified 5-Fold Cross Validation
# ------------------------------------------------------------

cv_rf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# GridSearchCV
# ------------------------------------------------------------

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    scoring="f1_macro",
    cv=cv_rf,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

# ------------------------------------------------------------
# Pencarian hyperparameter
# HANYA menggunakan data training
# ------------------------------------------------------------

grid_rf.fit(
    X_train_rf,
    y_train_rf
)

# ------------------------------------------------------------
# Menampilkan hasil terbaik
# ------------------------------------------------------------

print("=" * 70)
print("HASIL PENENTUAN HYPERPARAMETER RANDOM FOREST")
print("=" * 70)

print("\nHyperparameter terbaik:")

for parameter, value in grid_rf.best_params_.items():
    print(f"{parameter:20s}: {value}")

print(
    "\nRata-rata Macro F1 CV :",
    round(grid_rf.best_score_, 4)
)

### 4.1.3 Inspeksi Decision Tree Random Forest

In [ ]:
# ============================================================
# INSPEKSI DECISION TREE PADA RANDOM FOREST TERBAIK
# ============================================================

best_rf = grid_rf.best_estimator_

print("=" * 70)
print("INFORMASI RANDOM FOREST TERBAIK")
print("=" * 70)

print("\nJumlah decision tree :", len(best_rf.estimators_))

# ------------------------------------------------------------
# Mengambil decision tree pertama
# ------------------------------------------------------------

tree_1 = best_rf.estimators_[0]

print("\nInformasi Decision Tree ke-1")
print("Kedalaman tree       :", tree_1.get_depth())
print("Jumlah leaf           :", tree_1.get_n_leaves())
print("Jumlah node           :", tree_1.tree_.node_count)

# ------------------------------------------------------------
# Informasi fitur yang digunakan
# ------------------------------------------------------------

print("\nFitur yang digunakan:")
for i, fitur in enumerate(X_train_rf.columns):
    print(f"{i}. {fitur}")

### 4.1.4 Visualisasi Decision Tree

In [ ]:
# ============================================================
# VISUALISASI STRUKTUR DECISION TREE UNTUK BAB III
# ============================================================

from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Mengambil decision tree pertama dari Random Forest terbaik
tree_1 = grid_rf.best_estimator_.estimators_[0]

plt.figure(figsize=(18, 8))

plot_tree(
    tree_1,
    feature_names=X_train_rf.columns,
    class_names=["Normal", "Stunting"],
    filled=True,
    max_depth=2,
    fontsize=9,
    proportion=True
)

plt.title(
    "Ilustrasi Struktur Decision Tree pada Random Forest"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALISASI DECISION TREE PERTAMA
# ============================================================

from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

tree_1 = grid_rf.best_estimator_.estimators_[0]

plt.figure(figsize=(24, 12))

plot_tree(
    tree_1,
    feature_names=X_train_rf.columns,
    class_names=["Normal", "Stunting"],
    filled=True,
    max_depth=3,
    fontsize=8
)

plt.title(
    "Contoh Struktur Decision Tree Pertama pada Random Forest"
)

plt.show()

### 4.1.5 Analisis Kandidat Root Node Random Forest

In [ ]:
# ============================================================
# ANALISIS KANDIDAT ROOT NODE RANDOM FOREST
# Membandingkan 5 fitur berdasarkan Gini Impurity
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. MENGAMBIL MODEL RANDOM FOREST TERBAIK
# ============================================================

# Mengambil model terbaik dari hasil GridSearchCV
rf = grid_rf.best_estimator_

# Menggunakan data training yang sama dengan saat GridSearchCV
X = X_train_rf.copy()
y = y_train_rf.copy()


# ------------------------------------------------------------
# Memastikan index X dan y sejajar
# ------------------------------------------------------------

X = X.reset_index(drop=True)
y = pd.Series(y).reset_index(drop=True)


# ============================================================
# 2. DAFTAR FITUR
# ============================================================

fitur = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "LiLA"
]


# ============================================================
# 3. VERIFIKASI MODEL DAN DATA
# ============================================================

print("=" * 70)
print("VERIFIKASI MODEL RANDOM FOREST")
print("=" * 70)

print("Jumlah decision tree :", len(rf.estimators_))
print("Jumlah fitur model   :", rf.n_features_in_)
print("Jumlah data training :", len(X))

print("\nFitur yang digunakan:")
for i, nama_fitur in enumerate(fitur, start=1):
    print(f"{i}. {nama_fitur}")


# ============================================================
# 4. MENGAMBIL DATA BOOTSTRAP TREE PERTAMA
# ============================================================

tree_index = 0

# Indeks sampel bootstrap yang digunakan
# oleh decision tree pertama
sample_indices = rf.estimators_samples_[tree_index]

X_bootstrap = X.iloc[
    sample_indices
].reset_index(drop=True)

y_bootstrap = y.iloc[
    sample_indices
].reset_index(drop=True)


print("\n" + "=" * 70)
print("DATA BOOTSTRAP DECISION TREE PERTAMA")
print("=" * 70)

print(
    "Jumlah sampel bootstrap :",
    len(X_bootstrap)
)

print("\nDistribusi kelas:")
print(
    y_bootstrap.value_counts()
)


# ============================================================
# 5. FUNGSI GINI IMPURITY
# ============================================================

def gini_impurity(y_data):

    # Jika tidak ada data
    if len(y_data) == 0:
        return 0.0

    # Menghitung proporsi setiap kelas
    proporsi = y_data.value_counts(
        normalize=True
    )

    # Rumus Gini:
    # Gini = 1 - Σ(pi²)
    return 1 - np.sum(
        proporsi ** 2
    )


# ============================================================
# 6. FUNGSI MENCARI SPLIT TERBAIK
# ============================================================

def cari_split_terbaik(
    X_data,
    y_data,
    daftar_fitur
):

    hasil = []

    # --------------------------------------------------------
    # Memeriksa setiap fitur
    # --------------------------------------------------------

    for nama_fitur in daftar_fitur:

        # Mengambil seluruh nilai unik fitur
        nilai = np.sort(
            X_data[nama_fitur].unique()
        )

        # Jika hanya memiliki satu nilai,
        # tidak dapat digunakan sebagai split
        if len(nilai) < 2:
            continue

        # ----------------------------------------------------
        # Threshold berada di antara dua nilai berurutan
        # ----------------------------------------------------

        thresholds = (
            nilai[:-1] + nilai[1:]
        ) / 2

        # Nilai awal
        gini_terbaik = np.inf
        threshold_terbaik = None

        jumlah_kiri_terbaik = None
        jumlah_kanan_terbaik = None

        gini_kiri_terbaik = None
        gini_kanan_terbaik = None

        # ----------------------------------------------------
        # Menguji seluruh threshold
        # ----------------------------------------------------

        for threshold in thresholds:

            # Cabang kiri
            kiri = (
                X_data[nama_fitur]
                <= threshold
            )

            # Cabang kanan
            kanan = ~kiri

            y_kiri = y_data[kiri]
            y_kanan = y_data[kanan]

            # Pastikan kedua cabang memiliki data
            if (
                len(y_kiri) == 0
                or len(y_kanan) == 0
            ):
                continue

            # ------------------------------------------------
            # Gini masing-masing cabang
            # ------------------------------------------------

            gini_kiri = gini_impurity(
                y_kiri
            )

            gini_kanan = gini_impurity(
                y_kanan
            )

            # ------------------------------------------------
            # Menghitung Gini setelah split
            # ------------------------------------------------

            n = len(y_data)

            n_kiri = len(y_kiri)
            n_kanan = len(y_kanan)

            gini_split = (
                (n_kiri / n) * gini_kiri
                +
                (n_kanan / n) * gini_kanan
            )

            # ------------------------------------------------
            # Menyimpan split terbaik
            # ------------------------------------------------

            if gini_split < gini_terbaik:

                gini_terbaik = gini_split

                threshold_terbaik = threshold

                jumlah_kiri_terbaik = n_kiri
                jumlah_kanan_terbaik = n_kanan

                gini_kiri_terbaik = gini_kiri
                gini_kanan_terbaik = gini_kanan

        # ----------------------------------------------------
        # Menyimpan hasil fitur
        # ----------------------------------------------------

        hasil.append({

            "Fitur":
                nama_fitur,

            "Threshold Terbaik":
                threshold_terbaik,

            "Gini Split":
                gini_terbaik,

            "Gini Kiri":
                gini_kiri_terbaik,

            "Gini Kanan":
                gini_kanan_terbaik,

            "Data Kiri":
                jumlah_kiri_terbaik,

            "Data Kanan":
                jumlah_kanan_terbaik
        })

    return pd.DataFrame(
        hasil
    )


# ============================================================
# 7. PERHITUNGAN UNTUK SELURUH 5 FITUR
# ============================================================

hasil_root = cari_split_terbaik(
    X_bootstrap,
    y_bootstrap,
    fitur
)


# ============================================================
# 8. MENGURUTKAN BERDASARKAN GINI TERKECIL
# ============================================================

hasil_root = hasil_root.sort_values(
    by="Gini Split",
    ascending=True
).reset_index(
    drop=True
)


# ============================================================
# 9. MENAMPILKAN HASIL PERBANDINGAN
# ============================================================

print("\n" + "=" * 70)
print("PERBANDINGAN KANDIDAT ROOT NODE")
print("=" * 70)

display(
    hasil_root.round(4)
)


# ============================================================
# 10. MENENTUKAN KANDIDAT ROOT NODE TERBAIK
# ============================================================

terbaik = hasil_root.iloc[0]

print("\n" + "=" * 70)
print("KANDIDAT ROOT NODE TERBAIK")
print("=" * 70)

print(
    "Fitur              :",
    terbaik["Fitur"]
)

print(
    "Threshold terbaik  :",
    round(
        terbaik["Threshold Terbaik"],
        4
    )
)

print(
    "Gini Split         :",
    round(
        terbaik["Gini Split"],
        4
    )
)

print(
    "Gini Kiri          :",
    round(
        terbaik["Gini Kiri"],
        4
    )
)

print(
    "Gini Kanan         :",
    round(
        terbaik["Gini Kanan"],
        4
    )
)

print(
    "Data Kiri          :",
    terbaik["Data Kiri"]
)

print(
    "Data Kanan         :",
    terbaik["Data Kanan"]
)


# ============================================================
# 11. MEMERIKSA ROOT NODE AKTUAL TREE PERTAMA
# ============================================================

tree = rf.estimators_[0]

# Indeks fitur root node
root_feature_index = (
    tree.tree_.feature[0]
)

# Threshold root node
root_threshold = (
    tree.tree_.threshold[0]
)

# Nama fitur root node
root_feature_name = fitur[
    root_feature_index
]


print("\n" + "=" * 70)
print("ROOT NODE AKTUAL DECISION TREE PERTAMA")
print("=" * 70)

print(
    "Fitur              :",
    root_feature_name
)

print(
    "Threshold          :",
    round(
        root_threshold,
        4
    )
)


# ============================================================
# 12. INFORMASI TREE PERTAMA
# ============================================================

print("\n" + "=" * 70)
print("INFORMASI DECISION TREE PERTAMA")
print("=" * 70)

print(
    "Kedalaman tree     :",
    tree.tree_.max_depth
)

print(
    "Jumlah leaf        :",
    tree.tree_.n_leaves
)

print(
    "Jumlah node        :",
    tree.tree_.node_count
)

### 4.1.6 Majority Voting Random Forest

In [ ]:
# ============================================================
# 37. CONTOH PROSES MAJORITY VOTING RANDOM FOREST
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Mengambil Random Forest terbaik
# ------------------------------------------------------------

best_rf = grid_rf.best_estimator_

# ------------------------------------------------------------
# Mengambil seluruh data testing dalam bentuk array
# agar sesuai dengan format data saat training tree
# ------------------------------------------------------------

X_test_array = X_test_rf.to_numpy()

# ------------------------------------------------------------
# Mengambil prediksi dari seluruh decision tree
# ------------------------------------------------------------

prediksi_semua_tree = np.array([
    tree.predict(X_test_array)
    for tree in best_rf.estimators_
])

# Bentuk:
# jumlah_tree x jumlah_data_testing
print("Bentuk hasil prediksi seluruh tree:",
      prediksi_semua_tree.shape)

# ------------------------------------------------------------
# Mencari satu data testing yang memiliki suara campuran
# ------------------------------------------------------------

index_contoh = None

for i in range(len(X_test_rf)):

    jumlah_normal = np.sum(
        prediksi_semua_tree[:, i] == 0
    )

    jumlah_stunting = np.sum(
        prediksi_semua_tree[:, i] == 1
    )

    # Mencari data yang diprediksi oleh kedua kelas
    if jumlah_normal > 0 and jumlah_stunting > 0:
        index_contoh = i
        break

# ------------------------------------------------------------
# Memastikan data campuran ditemukan
# ------------------------------------------------------------

if index_contoh is None:

    print(
        "Tidak ditemukan data dengan suara campuran."
    )

else:

    # --------------------------------------------------------
    # Data contoh
    # --------------------------------------------------------

    data_contoh = X_test_rf.iloc[[index_contoh]]

    target_asli = y_test_rf.iloc[index_contoh]

    prediksi_tree_contoh = prediksi_semua_tree[
        :,
        index_contoh
    ]

    # --------------------------------------------------------
    # Menghitung suara
    # --------------------------------------------------------

    jumlah_normal = np.sum(
        prediksi_tree_contoh == 0
    )

    jumlah_stunting = np.sum(
        prediksi_tree_contoh == 1
    )

    # --------------------------------------------------------
    # Majority voting
    # --------------------------------------------------------

    if jumlah_normal > jumlah_stunting:
        hasil_voting = 0
    else:
        hasil_voting = 1

    nama_kelas = {
        0: "Normal",
        1: "Stunting"
    }

    # --------------------------------------------------------
    # Menampilkan hasil
    # --------------------------------------------------------

    print("=" * 70)
    print("CONTOH MAJORITY VOTING RANDOM FOREST")
    print("=" * 70)

    print("\nIndex data testing :", index_contoh)

    print("\nData testing yang digunakan:")
    print(data_contoh)

    print(
        "\nTarget aktual :",
        nama_kelas[target_asli]
    )

    print(
        "\nJumlah decision tree :",
        len(best_rf.estimators_)
    )

    print("\nJumlah suara:")
    print(
        "Normal   :",
        jumlah_normal
    )
    print(
        "Stunting :",
        jumlah_stunting
    )

    print("\nPersentase suara:")
    print(
        "Normal   :",
        round(
            jumlah_normal /
            len(best_rf.estimators_) * 100,
            2
        ),
        "%"
    )

    print(
        "Stunting :",
        round(
            jumlah_stunting /
            len(best_rf.estimators_) * 100,
            2
        ),
        "%"
    )

    print(
        "\nHasil majority voting :",
        nama_kelas[hasil_voting]
    )

    print(
        "Prediksi sesuai target aktual :",
        hasil_voting == target_asli
    )

### 4.1.7 Pelatihan Model Random Forest Final

In [ ]:
# ============================================================
# 38. PELATIHAN MODEL RANDOM FOREST FINAL
# ============================================================

from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# Membentuk model Random Forest final
# berdasarkan hyperparameter terbaik
# ------------------------------------------------------------

rf_final = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    max_features="sqrt",
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# Melatih model menggunakan seluruh data training
# hasil RUS 3:1 + Regular SMOTE
# ------------------------------------------------------------

rf_final.fit(
    X_train_rf,
    y_train_rf
)

# ------------------------------------------------------------
# Verifikasi model
# ------------------------------------------------------------

print("=" * 70)
print("PELATIHAN RANDOM FOREST FINAL")
print("=" * 70)

print("\nJumlah data training :", X_train_rf.shape[0])
print("Jumlah fitur         :", X_train_rf.shape[1])

print("\nJumlah decision tree :",
      len(rf_final.estimators_))

print("\nHyperparameter model:")
print("criterion          :", rf_final.criterion)
print("max_depth          :", rf_final.max_depth)
print("max_features       :", rf_final.max_features)
print("min_samples_split  :", rf_final.min_samples_split)
print("min_samples_leaf   :", rf_final.min_samples_leaf)
print("n_estimators       :", rf_final.n_estimators)

print("\nModel Random Forest final berhasil dilatih.")

In [ ]:
# ============================================================
# 40. CONFUSION MATRIX RANDOM FOREST
# ============================================================

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Menghitung confusion matrix
# ------------------------------------------------------------

cm_rf = confusion_matrix(
    y_test_rf,
    y_pred_rf
)

# ------------------------------------------------------------
# Mengambil nilai TN, FP, FN, TP
# ------------------------------------------------------------

TN, FP, FN, TP = cm_rf.ravel()

# ------------------------------------------------------------
# Menampilkan nilai confusion matrix
# ------------------------------------------------------------

print("=" * 70)
print("CONFUSION MATRIX RANDOM FOREST")
print("=" * 70)

print("\nConfusion Matrix:")
print(cm_rf)

print("\nKomponen Confusion Matrix:")
print("True Negative  (TN) :", TN)
print("False Positive (FP) :", FP)
print("False Negative (FN) :", FN)
print("True Positive  (TP) :", TP)

# ------------------------------------------------------------
# Verifikasi jumlah data
# ------------------------------------------------------------

print("\nVerifikasi:")
print(
    "TN + FP + FN + TP =",
    TN + FP + FN + TP
)

print(
    "Jumlah data testing =",
    len(y_test_rf)
)

In [ ]:
# ============================================================
# VISUALISASI CONFUSION MATRIX RANDOM FOREST
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# ------------------------------------------------------------
# Menampilkan confusion matrix
# ------------------------------------------------------------

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=["Normal", "Stunting"]
)

fig, ax = plt.subplots(figsize=(7, 6))

disp.plot(
    ax=ax,
    values_format="d",
    cmap="viridis",
    colorbar=True
)

plt.title("Confusion Matrix Random Forest")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 41. EVALUASI KINERJA RANDOM FOREST
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

accuracy_rf = accuracy_score(
    y_test_rf,
    y_pred_rf
)

# ------------------------------------------------------------
# Precision
# Macro Average
# ------------------------------------------------------------

precision_rf = precision_score(
    y_test_rf,
    y_pred_rf,
    average="macro"
)

# ------------------------------------------------------------
# Recall
# Macro Average
# ------------------------------------------------------------

recall_rf = recall_score(
    y_test_rf,
    y_pred_rf,
    average="macro"
)

# ------------------------------------------------------------
# F1-Score
# Macro Average
# ------------------------------------------------------------

f1_rf = f1_score(
    y_test_rf,
    y_pred_rf,
    average="macro"
)

# ------------------------------------------------------------
# Menampilkan hasil evaluasi
# ------------------------------------------------------------

print("=" * 70)
print("EVALUASI KINERJA RANDOM FOREST")
print("=" * 70)

print(
    f"\nAccuracy        : {accuracy_rf:.4f} "
    f"({accuracy_rf * 100:.2f}%)"
)

print(
    f"Precision Macro : {precision_rf:.4f} "
    f"({precision_rf * 100:.2f}%)"
)

print(
    f"Recall Macro    : {recall_rf:.4f} "
    f"({recall_rf * 100:.2f}%)"
)

print(
    f"F1-Score Macro  : {f1_rf:.4f} "
    f"({f1_rf * 100:.2f}%)"
)

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test_rf,
        y_pred_rf,
        target_names=[
            "Normal",
            "Stunting"
        ],
        digits=4
    )
)

## 4.2 Pemodelan K-Nearest Neighbor

### 4.2.1 Persiapan Data K-Nearest Neighbor

In [ ]:
# ============================================================
# 42. PERSIAPAN DATA K-NEAREST NEIGHBOR
# ============================================================

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Standardisasi fitur
# ------------------------------------------------------------

scaler_knn = StandardScaler()

# Fit hanya pada data training
X_train_knn = scaler_knn.fit_transform(
    X_train_rf
)

# Transform data testing menggunakan scaler training
X_test_knn = scaler_knn.transform(
    X_test_rf
)

# ------------------------------------------------------------
# Verifikasi ukuran data
# ------------------------------------------------------------

print("=" * 70)
print("PERSIAPAN DATA K-NEAREST NEIGHBOR")
print("=" * 70)

print("\nUkuran data training :", X_train_knn.shape)
print("Ukuran data testing  :", X_test_knn.shape)

print("\nJumlah fitur :", X_train_knn.shape[1])

print("\nFitur yang digunakan:")
for i, fitur in enumerate(X_train_rf.columns, start=1):
    print(f"{i}. {fitur}")

print("\nStandardisasi berhasil diterapkan.")

### 4.2.2 Penentuan Nilai K K-Nearest Neighbor

In [ ]:
# ============================================================
# 43. PENENTUAN NILAI K KNN DENGAN STRATIFIED 5-FOLD CV
# ============================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import pandas as pd

# ------------------------------------------------------------
# Kandidat nilai K
# ------------------------------------------------------------

nilai_k = [3, 5, 7, 9, 11, 13, 15]

# ------------------------------------------------------------
# Stratified 5-Fold Cross Validation
# ------------------------------------------------------------

cv_knn = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

hasil_k = []

print("=" * 70)
print("PENENTUAN NILAI K KNN")
print("=" * 70)

for k in nilai_k:

    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric="euclidean",
        weights="uniform"
    )

    skor = cross_val_score(
        knn,
        X_train_knn,
        y_train_rf,
        cv=cv_knn,
        scoring="f1_macro",
        n_jobs=-1
    )

    rata_rata = skor.mean()

    hasil_k.append({
        "K": k,
        "Fold 1": skor[0],
        "Fold 2": skor[1],
        "Fold 3": skor[2],
        "Fold 4": skor[3],
        "Fold 5": skor[4],
        "Macro F1": rata_rata
    })

    print(
        f"K = {k:2d} | "
        f"Fold 1 = {skor[0]:.4f} | "
        f"Fold 2 = {skor[1]:.4f} | "
        f"Fold 3 = {skor[2]:.4f} | "
        f"Fold 4 = {skor[3]:.4f} | "
        f"Fold 5 = {skor[4]:.4f} | "
        f"Macro F1 = {rata_rata:.4f}"
    )

# ------------------------------------------------------------
# Membuat tabel hasil
# ------------------------------------------------------------

hasil_k_df = pd.DataFrame(hasil_k)

# ------------------------------------------------------------
# Menentukan K terbaik
# ------------------------------------------------------------

index_terbaik = hasil_k_df["Macro F1"].idxmax()

k_terbaik = int(
    hasil_k_df.loc[index_terbaik, "K"]
)

f1_cv_knn_terbaik = (
    hasil_k_df.loc[index_terbaik, "Macro F1"]
)

# ------------------------------------------------------------
# Menampilkan hasil akhir
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HASIL PENENTUAN NILAI K")
print("=" * 70)

print(
    hasil_k_df[
        ["K", "Macro F1"]
    ].to_string(index=False)
)

print(
    "\nK terbaik :",
    k_terbaik
)

print(
    "Macro F1 CV :",
    round(f1_cv_knn_terbaik, 4)
)

In [ ]:
import matplotlib.pyplot as plt

# Data hasil pengujian kandidat nilai K
k_values = [3, 5, 7, 9, 11, 13, 15]
macro_f1 = [0.9653, 0.9583, 0.9490, 0.9371, 0.9296, 0.9248, 0.9246]

# Membuat grafik
plt.figure(figsize=(8, 5))
plt.bar(k_values, macro_f1)

# Judul dan label
plt.title('Perbandingan Macro F1-Score pada Kandidat Nilai K')
plt.xlabel('Nilai K')
plt.ylabel('Macro F1-Score')

# Menampilkan nilai di atas batang
for x, y in zip(k_values, macro_f1):
    plt.text(x, y + 0.003, f'{y:.4f}',
             ha='center', va='bottom')

# Skala sumbu
plt.xticks(k_values)
plt.ylim(0.90, 1.00)

plt.tight_layout()
plt.show()

### 4.2.3 Perhitungan Jarak Euclidean

In [ ]:
# ============================================================
# MENAMPILKAN 15 DATA TRAINING DENGAN JARAK TERDEKAT
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Memilih data testing yang sama dengan contoh perhitungan KNN
# ------------------------------------------------------------

index_contoh_knn = 2

data_uji = X_test_knn[index_contoh_knn]

# ------------------------------------------------------------
# Menghitung jarak Euclidean terhadap seluruh data training
# ------------------------------------------------------------

jarak = np.sqrt(
    np.sum(
        (X_train_knn - data_uji) ** 2,
        axis=1
    )
)

# ------------------------------------------------------------
# Membentuk tabel seluruh jarak
# ------------------------------------------------------------

data_jarak_15 = pd.DataFrame({

    "Index Training":
        X_train_rf.index,

    "Jarak Euclidean":
        jarak,

    "Kelas":
        y_train_rf.to_numpy()
})

# ------------------------------------------------------------
# Mengubah kode target menjadi nama kelas
# ------------------------------------------------------------

data_jarak_15["Kelas"] = (
    data_jarak_15["Kelas"]
    .map({
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Mengurutkan jarak dari terkecil ke terbesar
# ------------------------------------------------------------

data_jarak_15 = (
    data_jarak_15
    .sort_values(
        by="Jarak Euclidean"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Mengambil 15 tetangga terdekat
# ------------------------------------------------------------

tetangga_15 = data_jarak_15.head(15).copy()

# Menambahkan nomor urutan tetangga
tetangga_15.insert(
    0,
    "Urutan",
    range(1, len(tetangga_15) + 1)
)

# ------------------------------------------------------------
# Membulatkan jarak
# ------------------------------------------------------------

tetangga_15["Jarak Euclidean"] = (
    tetangga_15["Jarak Euclidean"]
    .round(6)
)

# ------------------------------------------------------------
# Menampilkan hasil
# ------------------------------------------------------------

print("=" * 75)
print("15 TETANGGA TERDEKAT DATA TESTING INDEKS 2")
print("=" * 75)

print(
    tetangga_15.to_string(
        index=False
    )
)

### 4.2.4 Pengurutan Tetangga Berdasarkan Jarak Euclidean

In [ ]:
# ============================================================
# 44. CONTOH PERHITUNGAN JARAK EUCLIDEAN DAN
#     TETANGGA TERDEKAT KNN
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Nilai K terbaik hasil Cross Validation
# ------------------------------------------------------------

k_knn = k_terbaik

# ------------------------------------------------------------
# Mencari data testing yang tidak memiliki
# jarak Euclidean = 0 terhadap data training
# ------------------------------------------------------------

index_contoh_knn = None
jarak_contoh_knn = None

for i in range(len(X_test_knn)):

    data_uji = X_test_knn[i]

    # Menghitung jarak Euclidean terhadap
    # seluruh data training
    jarak = np.sqrt(
        np.sum(
            (X_train_knn - data_uji) ** 2,
            axis=1
        )
    )

    # Memilih data testing yang tidak memiliki
    # data training dengan jarak = 0
    if np.min(jarak) > 0:

        index_contoh_knn = i
        jarak_contoh_knn = jarak

        break

# ------------------------------------------------------------
# Verifikasi apakah data contoh berhasil ditemukan
# ------------------------------------------------------------

if index_contoh_knn is None:

    print(
        "Tidak ditemukan data testing "
        "yang tidak memiliki jarak Euclidean = 0."
    )

else:

    # --------------------------------------------------------
    # Mengambil data testing contoh
    # --------------------------------------------------------

    data_uji = X_test_knn[index_contoh_knn]

    target_asli_knn = y_test_rf.iloc[
        index_contoh_knn
    ]

    # --------------------------------------------------------
    # Membentuk tabel jarak terhadap seluruh data training
    # --------------------------------------------------------

    data_jarak = pd.DataFrame({

        "Index Training":
            X_train_rf.index,

        "Jarak Euclidean":
            jarak_contoh_knn,

        "Kelas":
            y_train_rf.to_numpy()
    })

    # --------------------------------------------------------
    # Mengubah kode target menjadi nama kelas
    # --------------------------------------------------------

    data_jarak["Kelas"] = (
        data_jarak["Kelas"]
        .map({
            0: "Normal",
            1: "Stunting"
        })
    )

    # --------------------------------------------------------
    # Mengurutkan berdasarkan jarak terkecil
    # --------------------------------------------------------

    data_jarak = (
        data_jarak
        .sort_values(
            by="Jarak Euclidean"
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Mengambil K = 3 tetangga terdekat
    # --------------------------------------------------------

    tetangga_terdekat = (
        data_jarak
        .head(k_knn)
    )

    # --------------------------------------------------------
    # Menampilkan informasi data testing
    # --------------------------------------------------------

    print("=" * 70)
    print("CONTOH PERHITUNGAN JARAK EUCLIDEAN KNN")
    print("=" * 70)

    print(
        "\nNilai K terbaik :",
        k_knn
    )

    print(
        "Index data testing :",
        index_contoh_knn
    )

    print(
        "\nNilai fitur data testing setelah standardisasi:"
    )

    for nama_fitur, nilai in zip(
        X_train_rf.columns,
        data_uji
    ):

        print(
            f"{nama_fitur:15s}: {nilai:.6f}"
        )

    print(
        "\nTarget aktual :",
        "Normal"
        if target_asli_knn == 0
        else "Stunting"
    )

    # --------------------------------------------------------
    # Menampilkan 3 tetangga terdekat
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("3 TETANGGA TERDEKAT")
    print("=" * 70)

    print(
        tetangga_terdekat.to_string(
            index=False
        )
    )

    # --------------------------------------------------------
    # Majority Voting
    # --------------------------------------------------------

    jumlah_kelas = (
        tetangga_terdekat["Kelas"]
        .value_counts()
    )

    print("\n" + "=" * 70)
    print("HASIL MAJORITY VOTING")
    print("=" * 70)

    print(jumlah_kelas)

    # --------------------------------------------------------
    # Menentukan kelas prediksi
    # --------------------------------------------------------

    kelas_prediksi_knn = (
        jumlah_kelas.idxmax()
    )

    target_nama = (
        "Normal"
        if target_asli_knn == 0
        else "Stunting"
    )

    print(
        "\nPrediksi KNN :",
        kelas_prediksi_knn
    )

    print(
        "Target aktual :",
        target_nama
    )

    print(
        "Prediksi sesuai target :",
        kelas_prediksi_knn == target_nama
    )

### 4.2.5 Majority Voting pada Kandidat Nilai K

In [ ]:
# ============================================================
# 45. PERHITUNGAN JARAK EUCLIDEAN SECARA DETAIL
# ============================================================

# ------------------------------------------------------------
# Mengambil 3 tetangga terdekat dari hasil sebelumnya
# ------------------------------------------------------------

index_tetangga = tetangga_terdekat["Index Training"].tolist()

# ------------------------------------------------------------
# Mengambil posisi data training berdasarkan index
# ------------------------------------------------------------

posisi_tetangga = [
    X_train_rf.index.get_loc(idx)
    for idx in index_tetangga
]

# ------------------------------------------------------------
# Nama fitur
# ------------------------------------------------------------

nama_fitur = X_train_rf.columns.tolist()

# ------------------------------------------------------------
# Nilai data testing
# ------------------------------------------------------------

data_testing_detail = X_test_knn[index_contoh_knn]

print("=" * 70)
print("PERHITUNGAN JARAK EUCLIDEAN SECARA DETAIL")
print("=" * 70)

print("\nData testing yang digunakan:")
print(
    "Index testing :",
    index_contoh_knn
)

print("\nNilai data testing setelah standardisasi:")

for fitur, nilai in zip(
    nama_fitur,
    data_testing_detail
):
    print(
        f"{fitur:15s}: {nilai:.6f}"
    )

# ------------------------------------------------------------
# Perhitungan untuk masing-masing tetangga
# ------------------------------------------------------------

for urutan, (idx, posisi) in enumerate(
    zip(index_tetangga, posisi_tetangga),
    start=1
):

    data_training_detail = X_train_knn[posisi]

    selisih = (
        data_testing_detail -
        data_training_detail
    )

    kuadrat_selisih = (
        selisih ** 2
    )

    jumlah_kuadrat = (
        np.sum(kuadrat_selisih)
    )

    jarak = np.sqrt(
        jumlah_kuadrat
    )

    kelas = (
        y_train_rf.iloc[posisi]
    )

    nama_kelas = (
        "Normal"
        if kelas == 0
        else "Stunting"
    )

    print("\n" + "-" * 70)
    print(
        f"TETANGGA KE-{urutan}"
    )
    print("-" * 70)

    print(
        "Index training :",
        idx
    )

    print(
        "Kelas :",
        nama_kelas
    )

    print("\nNilai fitur training:")

    for fitur, nilai in zip(
        nama_fitur,
        data_training_detail
    ):
        print(
            f"{fitur:15s}: {nilai:.6f}"
        )

    print("\nSelisih kuadrat setiap fitur:")

    for fitur, nilai in zip(
        nama_fitur,
        kuadrat_selisih
    ):
        print(
            f"{fitur:15s}: {nilai:.8f}"
        )

    print(
        "\nJumlah kuadrat selisih :",
        f"{jumlah_kuadrat:.8f}"
    )

    print(
        "Jarak Euclidean        :",
        f"{jarak:.6f}"
    )

### 4.2.6 Pelatihan Model K-Nearest Neighbor Final

In [ ]:
# ============================================================
# 46. PELATIHAN MODEL KNN FINAL
# ============================================================

from sklearn.neighbors import KNeighborsClassifier

print("=" * 70)
print("PELATIHAN KNN FINAL")
print("=" * 70)

# ------------------------------------------------------------
# Membentuk model KNN final
# ------------------------------------------------------------

knn_final = KNeighborsClassifier(
    n_neighbors=k_terbaik,
    metric="euclidean",
    weights="uniform"
)

# ------------------------------------------------------------
# Melatih model menggunakan seluruh data training
# hasil RUS 3:1 + Regular SMOTE
# ------------------------------------------------------------

knn_final.fit(
    X_train_knn,
    y_train_rf
)

# ------------------------------------------------------------
# Informasi model
# ------------------------------------------------------------

print(
    "\nJumlah data training :",
    X_train_knn.shape[0]
)

print(
    "Jumlah fitur         :",
    X_train_knn.shape[1]
)

print(
    "Nilai K              :",
    k_terbaik
)

print(
    "Metric jarak         :",
    "Euclidean"
)

print(
    "Weights              :",
    "uniform"
)

print(
    "\nModel KNN final berhasil dilatih."
)

# 5. Evaluation

## 5.1 Prediksi Data Testing Random Forest

In [ ]:
# ============================================================
# 39. PREDIKSI DATA TESTING MENGGUNAKAN RANDOM FOREST
# ============================================================

# ------------------------------------------------------------
# Melakukan prediksi pada seluruh data testing
# ------------------------------------------------------------

y_pred_rf = rf_final.predict(
    X_test_rf
)

# ------------------------------------------------------------
# Menampilkan hasil prediksi
# ------------------------------------------------------------

print("=" * 70)
print("PREDIKSI DATA TESTING RANDOM FOREST")
print("=" * 70)

print("\nJumlah data testing :", len(X_test_rf))

print("\nJumlah hasil prediksi :", len(y_pred_rf))

print("\nDistribusi hasil prediksi:")

print(
    pd.Series(y_pred_rf)
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Membandingkan prediksi dengan target aktual
# ------------------------------------------------------------

print("\nDistribusi target aktual:")

print(
    y_test_rf
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Verifikasi jumlah prediksi
# ------------------------------------------------------------

print("\nVerifikasi jumlah data:")
print("Data testing :", len(X_test_rf))
print("Prediksi      :", len(y_pred_rf))

print(
    "\nJumlah data dan prediksi sama :",
    len(X_test_rf) == len(y_pred_rf)
)

## 5.2 Confusion Matrix Random Forest

## 5.3 Evaluasi Kinerja Random Forest

## 5.4 Prediksi Data Testing K-Nearest Neighbor

In [ ]:
# ============================================================
# 47. PREDIKSI DATA TESTING KNN
# ============================================================

print("=" * 70)
print("PREDIKSI DATA TESTING KNN")
print("=" * 70)

# ------------------------------------------------------------
# Melakukan prediksi terhadap seluruh data testing
# ------------------------------------------------------------

y_pred_knn = knn_final.predict(
    X_test_knn
)

# ------------------------------------------------------------
# Menampilkan jumlah hasil prediksi
# ------------------------------------------------------------

print(
    "\nJumlah data testing :",
    len(X_test_knn)
)

print(
    "Jumlah hasil prediksi :",
    len(y_pred_knn)
)

# ------------------------------------------------------------
# Distribusi hasil prediksi
# ------------------------------------------------------------

prediksi_knn_series = pd.Series(
    y_pred_knn,
    name="Target"
)

print("\nDistribusi hasil prediksi:")

print(
    prediksi_knn_series
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Distribusi target aktual
# ------------------------------------------------------------

print("\nDistribusi target aktual:")

print(
    y_test_rf
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Normal",
        1: "Stunting"
    })
)

# ------------------------------------------------------------
# Verifikasi jumlah data
# ------------------------------------------------------------

print("\nVerifikasi jumlah data:")

print(
    "Data testing :",
    len(X_test_knn)
)

print(
    "Prediksi      :",
    len(y_pred_knn)
)

print(
    "Jumlah data dan prediksi sama :",
    len(X_test_knn) == len(y_pred_knn)
)

## 5.5 Confusion Matrix K-Nearest Neighbor

In [ ]:
# ============================================================
# 48. CONFUSION MATRIX KNN
# ============================================================

from sklearn.metrics import confusion_matrix

print("=" * 70)
print("CONFUSION MATRIX KNN")
print("=" * 70)

# ------------------------------------------------------------
# Menghitung confusion matrix
# ------------------------------------------------------------

cm_knn = confusion_matrix(
    y_test_rf,
    y_pred_knn
)

print("\nConfusion Matrix:")
print(cm_knn)

# ------------------------------------------------------------
# Mengambil komponen confusion matrix
# ------------------------------------------------------------

tn_knn, fp_knn, fn_knn, tp_knn = cm_knn.ravel()

print("\nKomponen Confusion Matrix:")

print(
    "True Negative  (TN) :",
    tn_knn
)

print(
    "False Positive (FP) :",
    fp_knn
)

print(
    "False Negative (FN) :",
    fn_knn
)

print(
    "True Positive  (TP) :",
    tp_knn
)

# ------------------------------------------------------------
# Verifikasi jumlah data
# ------------------------------------------------------------

total_cm_knn = (
    tn_knn +
    fp_knn +
    fn_knn +
    tp_knn
)

print("\nVerifikasi:")

print(
    "TN + FP + FN + TP =",
    total_cm_knn
)

print(
    "Jumlah data testing =",
    len(y_test_rf)
)

print(
    "Verifikasi jumlah data :",
    total_cm_knn == len(y_test_rf)
)

## 5.6 Visualisasi Confusion Matrix K-Nearest Neighbor

In [ ]:
# ============================================================
# 49. VISUALISASI CONFUSION MATRIX KNN
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

print("=" * 70)
print("VISUALISASI CONFUSION MATRIX KNN")
print("=" * 70)

# ------------------------------------------------------------
# Membuat visualisasi confusion matrix
# ------------------------------------------------------------

disp_knn = ConfusionMatrixDisplay(
    confusion_matrix=cm_knn,
    display_labels=[
        "Normal",
        "Stunting"
    ]
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

disp_knn.plot(
    ax=ax,
    values_format="d"
)

ax.set_title(
    "Confusion Matrix K-Nearest Neighbor"
)

ax.set_xlabel(
    "Predicted Label"
)

ax.set_ylabel(
    "True Label"
)

plt.tight_layout()
plt.show()

## 5.7 Evaluasi Kinerja K-Nearest Neighbor

In [ ]:
# ============================================================
# 50. EVALUASI KINERJA K-NEAREST NEIGHBOR
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("=" * 70)
print("EVALUASI KINERJA K-NEAREST NEIGHBOR")
print("=" * 70)

# ------------------------------------------------------------
# Menghitung metrik evaluasi
# ------------------------------------------------------------

accuracy_knn = accuracy_score(
    y_test_rf,
    y_pred_knn
)

precision_knn = precision_score(
    y_test_rf,
    y_pred_knn,
    average="macro"
)

recall_knn = recall_score(
    y_test_rf,
    y_pred_knn,
    average="macro"
)

f1_knn = f1_score(
    y_test_rf,
    y_pred_knn,
    average="macro"
)

# ------------------------------------------------------------
# Menampilkan hasil
# ------------------------------------------------------------

print(
    f"\nAccuracy        : "
    f"{accuracy_knn:.4f} "
    f"({accuracy_knn * 100:.2f}%)"
)

print(
    f"Precision Macro : "
    f"{precision_knn:.4f} "
    f"({precision_knn * 100:.2f}%)"
)

print(
    f"Recall Macro    : "
    f"{recall_knn:.4f} "
    f"({recall_knn * 100:.2f}%)"
)

print(
    f"F1-Score Macro  : "
    f"{f1_knn:.4f} "
    f"({f1_knn * 100:.2f}%)"
)

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test_rf,
        y_pred_knn,
        target_names=[
            "Normal",
            "Stunting"
        ],
        digits=4
    )
)

## 5.8 Perbandingan Kinerja Random Forest dan K-Nearest Neighbor

In [ ]:
# ============================================================
# 51. PERBANDINGAN KINERJA RANDOM FOREST DAN KNN
# ============================================================

import pandas as pd

print("=" * 70)
print("PERBANDINGAN KINERJA RANDOM FOREST DAN KNN")
print("=" * 70)

# ------------------------------------------------------------
# Membentuk tabel perbandingan
# ------------------------------------------------------------

hasil_perbandingan = pd.DataFrame({
    "Model": [
        "Random Forest",
        "KNN"
    ],

    "Accuracy": [
        accuracy_rf,
        accuracy_knn
    ],

    "Precision Macro": [
        precision_rf,
        precision_knn
    ],

    "Recall Macro": [
        recall_rf,
        recall_knn
    ],

    "F1-Score Macro": [
        f1_rf,
        f1_knn
    ]
})

# ------------------------------------------------------------
# Menampilkan dalam bentuk persentase
# ------------------------------------------------------------

hasil_persentase = hasil_perbandingan.copy()

kolom_metrik = [
    "Accuracy",
    "Precision Macro",
    "Recall Macro",
    "F1-Score Macro"
]

for kolom in kolom_metrik:
    hasil_persentase[kolom] = (
        hasil_persentase[kolom] * 100
    ).round(2)

print("\nHasil Perbandingan:")

print(
    hasil_persentase.to_string(
        index=False
    )
)

## 5.9 Visualisasi Perbandingan Kinerja Model

In [ ]:
# ============================================================
# 52. VISUALISASI PERBANDINGAN KINERJA RANDOM FOREST DAN KNN
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Data metrik
# ------------------------------------------------------------

model = [
    "Random Forest",
    "KNN"
]

accuracy = [
    accuracy_rf * 100,
    accuracy_knn * 100
]

precision = [
    precision_rf * 100,
    precision_knn * 100
]

recall = [
    recall_rf * 100,
    recall_knn * 100
]

f1 = [
    f1_rf * 100,
    f1_knn * 100
]

metrik = [
    "Accuracy",
    "Precision Macro",
    "Recall Macro",
    "F1-Score Macro"
]

nilai_rf = [
    accuracy[0],
    precision[0],
    recall[0],
    f1[0]
]

nilai_knn = [
    accuracy[1],
    precision[1],
    recall[1],
    f1[1]
]

# ------------------------------------------------------------
# Membuat posisi batang
# ------------------------------------------------------------

x = np.arange(len(metrik))
lebar = 0.35

fig, ax = plt.subplots(
    figsize=(10, 6)
)

# ------------------------------------------------------------
# Membuat bar chart
# ------------------------------------------------------------

bar_rf = ax.bar(
    x - lebar / 2,
    nilai_rf,
    lebar,
    label="Random Forest"
)

bar_knn = ax.bar(
    x + lebar / 2,
    nilai_knn,
    lebar,
    label="KNN"
)

# ------------------------------------------------------------
# Label dan judul
# ------------------------------------------------------------

ax.set_title(
    "Perbandingan Kinerja Random Forest dan KNN"
)

ax.set_ylabel(
    "Nilai (%)"
)

ax.set_xticks(x)

ax.set_xticklabels(
    metrik
)

ax.set_ylim(
    0,
    105
)

ax.legend()

# ------------------------------------------------------------
# Menampilkan nilai di atas batang
# ------------------------------------------------------------

for bar in bar_rf:

    tinggi = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        tinggi + 0.5,
        f"{tinggi:.2f}%",
        ha="center",
        va="bottom"
    )


for bar in bar_knn:

    tinggi = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        tinggi + 0.5,
        f"{tinggi:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

## 5.11 Penentuan Model Terbaik

In [ ]:
# ============================================================
# 53. PENENTUAN MODEL TERBAIK
# ============================================================

print("=" * 70)
print("PENENTUAN MODEL TERBAIK")
print("=" * 70)

# ------------------------------------------------------------
# Menentukan model berdasarkan seluruh metrik
# ------------------------------------------------------------

metrik_rf = [
    accuracy_rf,
    precision_rf,
    recall_rf,
    f1_rf
]

metrik_knn = [
    accuracy_knn,
    precision_knn,
    recall_knn,
    f1_knn
]

jumlah_unggul_rf = sum(
    rf > knn
    for rf, knn in zip(
        metrik_rf,
        metrik_knn
    )
)

jumlah_unggul_knn = sum(
    knn > rf
    for rf, knn in zip(
        metrik_rf,
        metrik_knn
    )
)

# ------------------------------------------------------------
# Menampilkan hasil
# ------------------------------------------------------------

print("\nJumlah metrik yang unggul:")

print(
    "Random Forest :",
    jumlah_unggul_rf,
    "dari 4 metrik"
)

print(
    "KNN           :",
    jumlah_unggul_knn,
    "dari 4 metrik"
)

# ------------------------------------------------------------
# Menentukan model terbaik
# ------------------------------------------------------------

if jumlah_unggul_rf > jumlah_unggul_knn:

    model_terbaik = "Random Forest"

elif jumlah_unggul_knn > jumlah_unggul_rf:

    model_terbaik = "K-Nearest Neighbor"

else:

    model_terbaik = "Tidak ada model yang dominan"

print(
    "\nModel terbaik :",
    model_terbaik
)

# ------------------------------------------------------------
# Menampilkan seluruh nilai
# ------------------------------------------------------------

print("\nNilai evaluasi:")

print(
    f"Random Forest - "
    f"Accuracy: {accuracy_rf * 100:.2f}% | "
    f"Precision Macro: {precision_rf * 100:.2f}% | "
    f"Recall Macro: {recall_rf * 100:.2f}% | "
    f"F1 Macro: {f1_rf * 100:.2f}%"
)

print(
    f"KNN - "
    f"Accuracy: {accuracy_knn * 100:.2f}% | "
    f"Precision Macro: {precision_knn * 100:.2f}% | "
    f"Recall Macro: {recall_knn * 100:.2f}% | "
    f"F1 Macro: {f1_knn * 100:.2f}%"
)

In [ ]:
# ============================================================
# 54. FEATURE IMPORTANCE RANDOM FOREST
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("FEATURE IMPORTANCE RANDOM FOREST")
print("=" * 70)

# ------------------------------------------------------------
# Mengambil feature importance dari model Random Forest final
# ------------------------------------------------------------

feature_importance_rf = pd.DataFrame({
    "Fitur": X_train_selected.columns,
    "Importance": rf_final.feature_importances_
})

# ------------------------------------------------------------
# Mengurutkan dari importance terbesar
# ------------------------------------------------------------

feature_importance_rf = (
    feature_importance_rf
    .sort_values(
        by="Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Menampilkan ranking
# ------------------------------------------------------------

print("\nRanking Feature Importance:")

for i, row in feature_importance_rf.iterrows():

    print(
        f"{i + 1}. "
        f"{row['Fitur']} : "
        f"{row['Importance']:.6f} "
        f"({row['Importance'] * 100:.2f}%)"
    )

# ------------------------------------------------------------
# Verifikasi jumlah importance
# ------------------------------------------------------------

total_importance = (
    feature_importance_rf["Importance"]
    .sum()
)

print(
    "\nTotal Feature Importance :",
    f"{total_importance:.6f}"
)

print(
    "Jumlah fitur :",
    len(feature_importance_rf)
)

In [ ]:
# ============================================================
# 55. VISUALISASI FEATURE IMPORTANCE RANDOM FOREST
# ============================================================

import matplotlib.pyplot as plt

print("=" * 70)
print("VISUALISASI FEATURE IMPORTANCE RANDOM FOREST")
print("=" * 70)

# ------------------------------------------------------------
# Data feature importance sudah diurutkan pada tahap sebelumnya
# ------------------------------------------------------------

fitur = feature_importance_rf["Fitur"]
importance = feature_importance_rf["Importance"] * 100

# ------------------------------------------------------------
# Membuat grafik
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

bars = plt.bar(
    fitur,
    importance
)

plt.title(
    "Feature Importance Random Forest"
)

plt.xlabel(
    "Fitur"
)

plt.ylabel(
    "Feature Importance (%)"
)

# Label fitur dibuat lurus/horizontal
plt.xticks(
    rotation=0
)

plt.ylim(
    0,
    max(importance) + 8
)

# ------------------------------------------------------------
# Menampilkan nilai persentase di atas batang
# ------------------------------------------------------------

for bar, nilai in zip(
    bars,
    importance
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{nilai:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ROC CURVE DAN AUC RANDOM FOREST & KNN
# MENGGUNAKAN 5 FITUR TERBARU
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# ------------------------------------------------------------
# 1. Fitur yang digunakan pada model final
# ------------------------------------------------------------

selected_features = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "LiLA"
]

# ------------------------------------------------------------
# 2. Ambil hanya 5 fitur dari data testing
# ------------------------------------------------------------

X_test_selected = X_test[selected_features].copy()

print("Fitur yang digunakan:")
print(X_test_selected.columns.tolist())

print("\nUkuran X_test:")
print(X_test_selected.shape)

# ------------------------------------------------------------
# 3. Probabilitas Random Forest
# ------------------------------------------------------------

y_prob_rf = rf_final.predict_proba(X_test_selected)[:, 1]

# ------------------------------------------------------------
# 4. Standardisasi data testing untuk KNN
# ------------------------------------------------------------
# Gunakan scaler yang sama dengan scaler saat training KNN

X_test_knn_scaled = scaler.transform(X_test_selected)

# ------------------------------------------------------------
# 5. Probabilitas KNN
# ------------------------------------------------------------

y_prob_knn = knn_final.predict_proba(X_test_knn_scaled)[:, 1]

# ------------------------------------------------------------
# 6. ROC Curve
# ------------------------------------------------------------

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)

# ------------------------------------------------------------
# 7. AUC
# ------------------------------------------------------------

auc_rf = roc_auc_score(y_test, y_prob_rf)
auc_knn = roc_auc_score(y_test, y_prob_knn)

print("\n" + "=" * 70)
print("HASIL ROC CURVE DAN AUC")
print("=" * 70)

print(f"AUC Random Forest : {auc_rf:.4f} ({auc_rf * 100:.2f}%)")
print(f"AUC KNN           : {auc_knn:.4f} ({auc_knn * 100:.2f}%)")

# ------------------------------------------------------------
# 8. Visualisasi
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC = {auc_rf:.4f})"
)

plt.plot(
    fpr_knn,
    tpr_knn,
    label=f"KNN (AUC = {auc_knn:.4f})"
)

# Garis random classifier
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("ROC Curve Random Forest dan K-Nearest Neighbor")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 56. PENYIMPANAN MODEL RANDOM FOREST FINAL
# ============================================================

import joblib
import os

print("=" * 70)
print("PENYIMPANAN MODEL RANDOM FOREST FINAL")
print("=" * 70)

# ------------------------------------------------------------
# Membuat folder model
# ------------------------------------------------------------

os.makedirs("models", exist_ok=True)

# ------------------------------------------------------------
# Menentukan nama file
# ------------------------------------------------------------

model_path = "models/random_forest_status_gizi.pkl"
feature_path = "models/fitur_random_forest.pkl"

# ------------------------------------------------------------
# Menyimpan model Random Forest final
# ------------------------------------------------------------

joblib.dump(
    rf_final,
    model_path
)

# ------------------------------------------------------------
# Menyimpan urutan fitur
# ------------------------------------------------------------

fitur_final = list(
    X_train_selected.columns
)

joblib.dump(
    fitur_final,
    feature_path
)

# ------------------------------------------------------------
# Verifikasi file
# ------------------------------------------------------------

print("\nModel berhasil disimpan:")
print(model_path)

print("\nFitur yang digunakan:")
for i, fitur in enumerate(fitur_final, start=1):
    print(f"{i}. {fitur}")

print("\nFile fitur berhasil disimpan:")
print(feature_path)

print("\nVerifikasi:")
print(
    "Jumlah fitur :",
    len(fitur_final)
)

print(
    "Model tersimpan :",
    os.path.exists(model_path)
)

print(
    "Fitur tersimpan :",
    os.path.exists(feature_path)
)

In [ ]:
# ============================================================
# MENYESUAIKAN X_TEST DENGAN 5 FITUR MODEL FINAL
# ============================================================

fitur_final = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "LiLA"
]

X_test_final = X_test[fitur_final].copy()

print("Fitur X_test sebelum seleksi:")
print(X_test.columns.tolist())

print("\nFitur yang digunakan model final:")
print(X_test_final.columns.tolist())

print("\nUkuran X_test_final:")
print(X_test_final.shape)

## 5.10 Evaluasi ROC Curve dan AUC

In [ ]:
# ============================================================
# ROC CURVE DAN AUC RANDOM FOREST & KNN
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# ------------------------------------------------------------
# Probabilitas kelas positif (Stunting = 1)
# ------------------------------------------------------------

y_prob_rf = rf_final.predict_proba(X_test_final)[:, 1]
y_prob_knn = knn_final.predict_proba(X_test_final)[:, 1]

# ------------------------------------------------------------
# Menghitung ROC Curve
# ------------------------------------------------------------

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)

# ------------------------------------------------------------
# Menghitung AUC
# ------------------------------------------------------------

auc_rf = roc_auc_score(y_test, y_prob_rf)
auc_knn = roc_auc_score(y_test, y_prob_knn)

print("=" * 70)
print("HASIL ROC CURVE DAN AUC")
print("=" * 70)

print(f"AUC Random Forest : {auc_rf:.4f} ({auc_rf * 100:.2f}%)")
print(f"AUC KNN           : {auc_knn:.4f} ({auc_knn * 100:.2f}%)")

# ------------------------------------------------------------
# Visualisasi ROC Curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC = {auc_rf:.4f})"
)

plt.plot(
    fpr_knn,
    tpr_knn,
    label=f"KNN (AUC = {auc_knn:.4f})"
)

# Garis baseline
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("ROC Curve Random Forest dan KNN")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Hasil Evaluasi Model Random Forest dan KNN
# Menjawab Rancangan Evaluasi pada Tabel BAB III
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Konfigurasi model terbaik
# ------------------------------------------------------------

konfigurasi_rf = (
    "n_estimators=200; "
    "max_depth=None; "
    "max_features=sqrt; "
    "min_samples_leaf=1; "
    "min_samples_split=2; "
    "criterion=gini"
)

konfigurasi_knn = (
    "n_neighbors=3; "
    "weights=distance; "
    "metric=euclidean"
)

# ------------------------------------------------------------
# Membentuk tabel hasil evaluasi
# ------------------------------------------------------------

tabel_hasil_evaluasi = pd.DataFrame({
    "No": [1, 2],

    "Algoritma": [
        "Random Forest",
        "K-Nearest Neighbor (KNN)"
    ],

    "Data Training": [
        "RUS 3:1 + Regular SMOTE",
        "RUS 3:1 + Regular SMOTE"
    ],

    "Konfigurasi Model": [
        konfigurasi_rf,
        konfigurasi_knn
    ],

    "Accuracy (%)": [
        accuracy_rf * 100,
        accuracy_knn * 100
    ],

    "Precision (%)": [
        precision_rf * 100,
        precision_knn * 100
    ],

    "Recall (%)": [
        recall_rf * 100,
        recall_knn * 100
    ],

    "F1-Score (%)": [
        f1_rf * 100,
        f1_knn * 100
    ],

    "AUC-ROC": [
        auc_rf,
        auc_knn
    ]
})

# ------------------------------------------------------------
# Membulatkan nilai
# ------------------------------------------------------------

kolom_persen = [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-Score (%)"
]

tabel_hasil_evaluasi[kolom_persen] = (
    tabel_hasil_evaluasi[kolom_persen].round(2)
)

tabel_hasil_evaluasi["AUC-ROC"] = (
    tabel_hasil_evaluasi["AUC-ROC"].round(4)
)

# ------------------------------------------------------------
# Menampilkan tabel hasil evaluasi
# ------------------------------------------------------------

print("=" * 100)
print("HASIL EVALUASI MODEL RANDOM FOREST DAN KNN")
print("=" * 100)

display(tabel_hasil_evaluasi)

# ------------------------------------------------------------
# Menampilkan model terpilih
# ------------------------------------------------------------

print("\nModel terpilih : Random Forest")

print(
    "Dasar pemilihan : Recall 100.00%, "
    "False Negative 0, AUC-ROC 0.9999, "
    "serta hasil evaluasi keseluruhan."
)

In [ ]:
# ============================================================
# Perbandingan Kinerja Random Forest dan KNN
# serta Penentuan Model Terbaik
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Mengambil nilai False Negative dari Confusion Matrix
# ------------------------------------------------------------

tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()
tn_knn, fp_knn, fn_knn, tp_knn = cm_knn.ravel()

# ------------------------------------------------------------
# Membentuk tabel perbandingan
# ------------------------------------------------------------

tabel_perbandingan_model = pd.DataFrame({
    "Model": [
        "Random Forest",
        "K-Nearest Neighbor"
    ],
    "Accuracy": [
        accuracy_rf * 100,
        accuracy_knn * 100
    ],
    "Precision": [
        precision_rf * 100,
        precision_knn * 100
    ],
    "Recall": [
        recall_rf * 100,
        recall_knn * 100
    ],
    "F1-Score": [
        f1_rf * 100,
        f1_knn * 100
    ],
    "AUC": [
        auc_rf,
        auc_knn
    ],
    "False Negative": [
        fn_rf,
        fn_knn
    ]
})

# ------------------------------------------------------------
# Membulatkan nilai
# ------------------------------------------------------------

tabel_perbandingan_model[
    ["Accuracy", "Precision", "Recall", "F1-Score"]
] = tabel_perbandingan_model[
    ["Accuracy", "Precision", "Recall", "F1-Score"]
].round(2)

tabel_perbandingan_model["AUC"] = (
    tabel_perbandingan_model["AUC"].round(4)
)

# ============================================================
# Menampilkan tabel perbandingan
# ============================================================

print("=" * 75)
print("PERBANDINGAN KINERJA RANDOM FOREST DAN KNN")
print("=" * 75)

display(tabel_perbandingan_model)

# ============================================================
# Penentuan Model Terbaik
# ============================================================

# Sesuai rancangan evaluasi penelitian:
# apabila terdapat perbedaan keunggulan antar-metrik,
# kemampuan mengenali kelas Stunting menjadi pertimbangan penting.
#
# Prioritas:
# 1. Recall lebih tinggi
# 2. False Negative lebih rendah
# 3. AUC lebih tinggi
# 4. F1-Score lebih tinggi
# 5. Precision lebih tinggi
# 6. Accuracy lebih tinggi

ranking_model = tabel_perbandingan_model.sort_values(
    by=[
        "Recall",
        "False Negative",
        "AUC",
        "F1-Score",
        "Precision",
        "Accuracy"
    ],
    ascending=[
        False,   # Recall lebih tinggi
        True,    # False Negative lebih rendah
        False,   # AUC lebih tinggi
        False,   # F1-Score lebih tinggi
        False,   # Precision lebih tinggi
        False    # Accuracy lebih tinggi
    ]
).reset_index(drop=True)

hasil_model_terbaik = ranking_model.iloc[0]

model_terbaik = hasil_model_terbaik["Model"]

# ============================================================
# Menampilkan Hasil Model Terbaik
# ============================================================

print("\n" + "=" * 75)
print("MODEL TERBAIK")
print("=" * 75)

print(
    "Model          :",
    model_terbaik
)

print(
    f"Accuracy       : "
    f"{hasil_model_terbaik['Accuracy']:.2f}%"
)

print(
    f"Precision      : "
    f"{hasil_model_terbaik['Precision']:.2f}%"
)

print(
    f"Recall         : "
    f"{hasil_model_terbaik['Recall']:.2f}%"
)

print(
    f"F1-Score       : "
    f"{hasil_model_terbaik['F1-Score']:.2f}%"
)

print(
    f"AUC            : "
    f"{hasil_model_terbaik['AUC']:.4f}"
)

print(
    f"False Negative : "
    f"{int(hasil_model_terbaik['False Negative'])}"
)

# ============================================================
# Kesimpulan
# ============================================================

print("\nKesimpulan:")

print(
    f"{model_terbaik} ditetapkan sebagai model terbaik "
    "berdasarkan hasil evaluasi keseluruhan dengan "
    "\nmempertimbangkan kemampuan model dalam mengenali "
    "kelas Stunting, nilai Recall, False Negative, AUC, "
    "F1-Score, Precision, dan Accuracy."
)

# 6. Deployment

## 6.2 Menyimpan Model Terbaik untuk Deployment

In [ ]:
# ============================================================
# MENYIMPAN MODEL RANDOM FOREST TERBAIK UNTUK DEPLOYMENT
# ============================================================

import joblib
import os
from google.colab import files

print("=" * 70)
print("MODEL RANDOM FOREST UNTUK DEPLOYMENT")
print("=" * 70)


# ------------------------------------------------------------
# 1. Model final
# ------------------------------------------------------------

model_deployment = rf_final


# ------------------------------------------------------------
# 2. Daftar fitur final penelitian
# ------------------------------------------------------------

fitur_deployment = [
    "JK",
    "Usia (Bulan)",
    "Berat",
    "Tinggi",
    "LiLA"
]


# ------------------------------------------------------------
# 3. Verifikasi jumlah fitur model
# ------------------------------------------------------------

if model_deployment.n_features_in_ != len(fitur_deployment):
    raise ValueError(
        "Jumlah fitur model tidak sesuai dengan fitur deployment. "
        f"Model membutuhkan {model_deployment.n_features_in_} fitur, "
        f"sedangkan fitur deployment berjumlah {len(fitur_deployment)}."
    )


# ------------------------------------------------------------
# 4. Menampilkan informasi model
# ------------------------------------------------------------

print("Algoritma     :", type(model_deployment).__name__)
print("Resampling    : RUS 3:1 + Regular SMOTE")
print("Jumlah fitur  :", model_deployment.n_features_in_)

print("\nUrutan fitur model:")

for nomor, fitur in enumerate(fitur_deployment, start=1):
    print(f"{nomor}. {fitur}")


# ------------------------------------------------------------
# 5. Menampilkan hyperparameter model
# ------------------------------------------------------------

print("\nHyperparameter Random Forest:")

print("n_estimators      :", model_deployment.n_estimators)
print("max_depth         :", model_deployment.max_depth)
print("max_features      :", model_deployment.max_features)
print("min_samples_leaf  :", model_deployment.min_samples_leaf)
print("min_samples_split :", model_deployment.min_samples_split)
print("criterion          :", model_deployment.criterion)


# ------------------------------------------------------------
# 6. Membentuk paket deployment
# ------------------------------------------------------------

paket_deployment = {
    "model": model_deployment,

    "fitur": fitur_deployment,

    "algoritma": "Random Forest",

    "resampling": "RUS 3:1 + Regular SMOTE",

    "n_estimators": model_deployment.n_estimators,

    "max_depth": model_deployment.max_depth,

    "max_features": model_deployment.max_features,

    "min_samples_leaf": model_deployment.min_samples_leaf,

    "min_samples_split": model_deployment.min_samples_split,

    "criterion": model_deployment.criterion,

    "accuracy": accuracy_rf,

    "precision": precision_rf,

    "recall": recall_rf,

    "f1_score": f1_rf,

    "auc": auc_rf,
}


# ------------------------------------------------------------
# 7. Menyimpan ke file PKL
# ------------------------------------------------------------

nama_file = "model_random_forest_stunting.pkl"

joblib.dump(
    paket_deployment,
    nama_file
)


# ------------------------------------------------------------
# 8. Verifikasi file
# ------------------------------------------------------------

print("\n" + "=" * 70)

if os.path.exists(nama_file):

    ukuran_file = os.path.getsize(nama_file)

    print("MODEL BERHASIL DISIMPAN")
    print("=" * 70)

    print("Nama file   :", nama_file)
    print("Ukuran file :", round(ukuran_file / 1024, 2), "KB")
    print("Jumlah fitur:", model_deployment.n_features_in_)

else:

    print("File model gagal dibuat.")


# ------------------------------------------------------------
# 9. Verifikasi isi file PKL
# ------------------------------------------------------------

paket_cek = joblib.load(nama_file)

print("\nVerifikasi isi PKL:")
print("Algoritma :", paket_cek["algoritma"])
print("Fitur     :", paket_cek["fitur"])
print("Jumlah    :", len(paket_cek["fitur"]))
print(
    "Model n_features_in_ :",
    paket_cek["model"].n_features_in_
)


# ------------------------------------------------------------
# 10. Download file PKL
# ------------------------------------------------------------

files.download(nama_file)